# RAG Mastery Notebook
### Retrieval-Augmented Generation — Beginner to Advanced Research Level (2026 Edition)

This notebook is the hands-on companion to the **RAG Mastery** reference document (20 chapters).
Every chapter here mirrors a chapter there: read the document first for the *why*, then run the
code here for the *how*.

**What you'll find in every chapter:**
- 📘 **Concept recap** — a short reminder of the idea, so this notebook is usable on its own too
- 💻 **Runnable code** — self-contained, dependency-free (NumPy / stdlib only), so it runs with **no API keys**
- 🔌 **Production notes** — exactly which real library/API to swap in when you leave the notebook
- 🧪 **Try it yourself** — a short exercise at the end of each chapter, with a hidden solution

> **How to use this notebook:** Run cells top to bottom. Chapters 1–10 build a working "naive" RAG
> pipeline from scratch, one piece at a time. Chapters 11–15 layer in advanced and agentic
> architectures on top of that pipeline. Chapters 16–20 cover evaluation, the full taxonomy of RAG
> types, known limitations, production engineering, and a guided capstone project.

---

## Table of Contents

**Part 1 — Foundations**
1. Introduction to RAG — What, Why, When
2. Why LLMs Alone Fail — The Case for Retrieval
3. RAG Architecture Fundamentals
4. Document Loading & Chunking Strategies
5. Embeddings & Vector Representations

**Part 2 — Building a Working RAG System**
6. Vector Databases & Indexing
7. Building Your First End-to-End Naive RAG Pipeline
8. Retrieval Techniques — Sparse, Dense & Hybrid Search
9. Query Transformation & Reranking
10. Prompt Engineering & Context Construction for RAG

**Part 3 — Advanced Architectures**
11. Advanced Architectures I — Self-RAG, Corrective RAG (CRAG), Adaptive RAG
12. Advanced Architectures II — GraphRAG & Knowledge-Graph RAG
13. Agentic RAG — Multi-Step, Tool-Using, Iterative Retrieval
14. Multimodal & Structured-Data RAG
15. RAPTOR, Late-Interaction (ColBERT) & Long-Context vs RAG

**Part 4 — Evaluation, Limitations & Production**
16. Evaluating RAG — RAGAS, ARES, DeepEval, TruLens & Custom Metrics
17. The Complete Taxonomy of RAG Types
18. Limitations & Failure Modes of RAG Systems
19. Production RAG — Architecture, Scaling, Security, Governance, Cost
20. Research Frontiers & Capstone Project

---


In [1]:
# Global setup -- run this once at the start of your session.
# Everything in this notebook runs on NumPy + the Python standard library only.
# No API keys, no internet connection, and no GPU are required for any cell.

import numpy as np
import re
import math
import random
from collections import Counter, defaultdict

random.seed(7)
np.random.seed(7)

print("Environment ready -- NumPy", np.__version__)
print("We will build a complete RAG system from first principles, one chapter at a time.")


Environment ready -- NumPy 2.0.2
We will build a complete RAG system from first principles, one chapter at a time.


## Chapter 1 — Introduction to RAG: What, Why, When

**Retrieval-Augmented Generation (RAG)** couples a language model with an external knowledge
source — documents, databases, internal wikis, APIs — so that it can pull in relevant information
*before* generating an answer, rather than relying solely on facts memorized during training.

The idea was introduced by Lewis et al. (Meta AI, 2020, [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)),
who combined a pretrained sequence-to-sequence generator with a dense passage retriever trained
end-to-end on Wikipedia. Their key finding: a *smaller* model with the *right retrieved passage*
regularly beat much larger models trying to recall facts purely from their parameters.

### The three core stages

Every RAG system, no matter how advanced, is built from three stages:

1. **Indexing (offline)** — documents are collected, split into chunks, converted into numerical
   vectors ("embeddings"), and stored in a searchable index. This happens once, before any user
   ever asks a question.
2. **Retrieval (online)** — when a user asks a question, the system searches the index and pulls
   back the chunks most likely to contain the answer.
3. **Generation (online)** — the retrieved chunks are inserted into a prompt alongside the
   question, and a language model generates an answer grounded in that evidence.

Everything else in this notebook — hybrid search, reranking, self-correction, knowledge graphs,
agentic loops — is increasingly sophisticated engineering built around these same three stages.

### RAG vs. fine-tuning vs. long context

| Approach | What it changes | Best for | Main limitation |
|---|---|---|---|
| **RAG** | Nothing in the model; adds external context at inference time | Frequently changing or large private knowledge bases; citations | Adds latency & system complexity |
| **Fine-tuning** | The model's own weights | Teaching a style, format, or narrow skill | Expensive to update; can't reliably store large fact volumes |
| **Long context** | Nothing; stuffs more raw text into the prompt | Small, bounded document sets | Cost/latency scale with context size; "lost in the middle" |

Let's build the very first piece: a tiny toy corpus we'll reuse and grow throughout this notebook.


In [2]:
# Chapter 1 -- Our running example corpus
# We'll use this same small corpus throughout Chapters 1-10 so every technique is directly
# comparable. In Chapter 4 we'll grow it; for now, five short passages are enough to illustrate
# the core ideas.

CORPUS_V1 = [
    "RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.",
    "Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.",
    "Dense embeddings place semantically similar sentences close together in vector space, even with different wording.",
    "BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.",
    "A reranker takes the shortlist from initial retrieval and re-scores it with a more accurate but slower model.",
]

for i, doc in enumerate(CORPUS_V1):
    print(f"[{i}] {doc}")


[0] RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
[1] Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.
[2] Dense embeddings place semantically similar sentences close together in vector space, even with different wording.
[3] BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
[4] A reranker takes the shortlist from initial retrieval and re-scores it with a more accurate but slower model.


### Demonstrating the three stages end to end (preview)

Below is a *deliberately naive* preview of all three stages working together, using nothing more
sophisticated than counting shared words. It will produce a mediocre answer — that's the point.
Every later chapter replaces one piece of this with something better, and you'll be able to watch
the answer quality improve step by step.


In [3]:
# Chapter 1 -- A deliberately naive preview of retrieve-then-generate
# (We'll replace every piece of this with something better in later chapters.)

def naive_retrieve(query, corpus, k=1):
    # Rank documents by how many words they share with the query. Crude on purpose.
    query_words = set(query.lower().split())
    scored = []
    for doc in corpus:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k]]

def naive_generate(query, retrieved_chunks):
    # Stand-in for an LLM call -- just formats the retrieved evidence as an "answer".
    if not retrieved_chunks:
        return "I don't have relevant information to answer this."
    evidence = retrieved_chunks[0]
    return f'Based on retrieved evidence: "{evidence}"'

query = "How does chunking affect what gets retrieved?"
retrieved = naive_retrieve(query, CORPUS_V1, k=1)
answer = naive_generate(query, retrieved)

print("QUERY:    ", query)
print("RETRIEVED:", retrieved[0])
print("ANSWER:   ", answer)


QUERY:     How does chunking affect what gets retrieved?
RETRIEVED: Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.
ANSWER:    Based on retrieved evidence: "Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost."


**Notice what's fragile here:** `naive_retrieve` only works because the query happens to share
literal words with the right passage ("chunking", "retrieved"). Change the wording of the query
even slightly — ask about "splitting documents" instead of "chunking" — and this approach will
likely fail, because it has no notion of *meaning*, only exact word overlap. That gap is exactly
what embeddings (Chapter 5) exist to close.

> **🏭 Production note:** In a real system, none of these three stages is implemented by hand.
> Indexing uses a document loader + a real embedding model + a vector database (Chapters 4–6).
> Retrieval uses that same embedding model plus an ANN search library (Chapters 6, 8).
> Generation calls an actual LLM API (OpenAI, Anthropic, or a self-hosted open-weight model) with
> a carefully constructed prompt (Chapter 10). Every one of those swaps is noted at the point it
> becomes relevant.


### 🧪 Try it yourself

Add a new query below and see whether `naive_retrieve` finds the right passage. Try:
1. A query that shares exact words with `CORPUS_V1[3]` (the BM25 passage) — it should work.
2. A query that means the same thing but uses *different* words — e.g. ask about "old-fashioned
   keyword search" instead of "sparse, keyword-based ranking." Does it still find the right passage?

Write your query in the cell below and run it.


In [4]:
# TODO Your turn -- write a query and test naive_retrieve
my_query = "old-fashioned keyword search"  # <-- try changing this

result = naive_retrieve(my_query, CORPUS_V1, k=2)
for r in result:
    print("-", r)

# Did it find the BM25 passage (CORPUS_V1[3])? If not, why not?
# (Hint: check the word overlap between my_query and CORPUS_V1[3] directly.)


- RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
- Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

With the query `"old-fashioned keyword search"`, `naive_retrieve` will likely **fail** to surface
the BM25 passage, because that passage says *"sparse, keyword-based ranking"* — it shares only the
word "keyword" with the query, which usually isn't enough to win against other passages by pure
word-overlap count.

This is the central motivation for Chapter 5 (embeddings): a good embedding model would place
"old-fashioned keyword search" and "sparse, keyword-based ranking" close together in vector space
even though they share almost no exact words, because their *meaning* is similar.

Try it yourself:
```python
set("old-fashioned keyword search".split()) & set(CORPUS_V1[3].lower().split())
```
You'll see the overlap is razor-thin — usually just `{"keyword"}`.
</details>


## Chapter 2 — Why LLMs Alone Fail: The Case for Retrieval

LLMs have three structural weaknesses that RAG directly addresses:

1. **Knowledge cutoff** — a model only knows what existed in its training data up to a fixed date.
   Anything after that is invisible to it unless retrieval supplies it.
2. **Hallucination** — when a model doesn't know something, it does not reliably say so. It often
   generates a fluent, confident-sounding response that is simply wrong.
3. **No access to private data** — a model was never trained on your organization's internal
   reports, a hospital's patient records, or a company's proprietary codebase.

Think of a plain LLM as answering "closed-book" — like an exam with no notes, relying entirely on
memory. A RAG system is "open-book": the model is handed the relevant page before it has to answer.

Let's build a small simulation of a model's "training knowledge" (a fixed, limited dictionary) and
compare closed-book vs. open-book answering on the same question.


In [5]:
# Chapter 2 -- Simulating closed-book vs. retrieval-augmented ("open-book") answering
#
# FAKE_TRAINING_KNOWLEDGE stands in for what an LLM "knows" from training -- a small,
# fixed set of facts frozen at some cutoff date. EXTERNAL_DOCS stands in for a live,
# up-to-date knowledge base that a RAG system can search at query time.

FAKE_TRAINING_KNOWLEDGE = {
    "python": "Python is a general-purpose programming language created by Guido van Rossum.",
    "paris": "Paris is the capital of France.",
}

EXTERNAL_DOCS = [
    "Acme Corp's Q3-2026 revenue was $482M, up 14% year-over-year, driven by cloud subscriptions.",
    "Acme Corp's customer churn rate improved from 6.2% to 4.1% after launching the loyalty program.",
    "The GRF-2026 weather monitoring project selected ESP32-S3 plus Raspberry Pi 4B as its hardware base.",
]

def closed_book_answer(query):
    q = query.lower()
    for k, v in FAKE_TRAINING_KNOWLEDGE.items():
        if k in q:
            return v
    # This is the failure mode: no matching fact, so a real LLM would likely
    # fabricate a plausible-sounding but ungrounded answer instead of admitting it doesn't know.
    return "[NOT IN TRAINING DATA] -- a real model would likely GUESS or HALLUCINATE an answer here."

def open_book_answer(query, docs):
    # Same crude word-overlap 'retrieval' as Chapter 1 -- real retrieval arrives in Chapter 5.
    query_words = set(query.lower().split())
    scored = sorted(docs, key=lambda d: len(query_words & set(d.lower().split())), reverse=True)
    best = scored[0]
    return f'Grounded in retrieved evidence: "{best}"'

test_queries = [
    "What was Acme Corp's Q3 2026 revenue?",
    "What is Python?",
]

for q in test_queries:
    print("QUERY:", q)
    print("  Closed-book:", closed_book_answer(q))
    print("  Open-book:  ", open_book_answer(q, EXTERNAL_DOCS))
    print()


QUERY: What was Acme Corp's Q3 2026 revenue?
  Closed-book: [NOT IN TRAINING DATA] -- a real model would likely GUESS or HALLUCINATE an answer here.
  Open-book:   Grounded in retrieved evidence: "Acme Corp's Q3-2026 revenue was $482M, up 14% year-over-year, driven by cloud subscriptions."

QUERY: What is Python?
  Closed-book: Python is a general-purpose programming language created by Guido van Rossum.
  Open-book:   Grounded in retrieved evidence: "Acme Corp's Q3-2026 revenue was $482M, up 14% year-over-year, driven by cloud subscriptions."



**Takeaway:** For the revenue question, closed-book has *nothing* in `FAKE_TRAINING_KNOWLEDGE` to
answer with — a real LLM would have to fabricate a number, a classic hallucination. Open-book finds
a real, traceable source. For the Python question, closed-book actually *does* have the fact (it
was in the frozen "training set"), which illustrates an important nuance: RAG doesn't replace a
model's parametric knowledge, it supplements it for anything outside that frozen snapshot.

> **🏭 Production note:** Real "hallucination" is more subtle than a missing dictionary lookup —
> models generate fluent text token by token and have no explicit "I don't know" flag by default.
> That's exactly why Chapter 10 covers explicit prompt instructions telling the model to say it
> doesn't know when the retrieved context doesn't contain the answer, rather than falling back on
> its own possibly-wrong general knowledge.


### 🧪 Try it yourself

Add a third document to `EXTERNAL_DOCS` about a topic *not* in `FAKE_TRAINING_KNOWLEDGE`, then ask
`closed_book_answer` and `open_book_answer` about it. Confirm that open-book finds it while
closed-book can't. Then try asking about something that's in **neither** — what does `open_book_answer`
do? (Hint: it always returns *something*, even a bad match — this is a preview of a real failure
mode covered in Chapter 18.)


In [6]:
# TODO Your turn

# 1. Add a new fact to EXTERNAL_DOCS
my_docs = EXTERNAL_DOCS + [
    "The AIRTOMS railway delay-recovery model runs a two-stage Router plus Narrator LLM pipeline.",
]

# 2. Ask about it
print(open_book_answer("How does the railway delay recovery model work?", my_docs))

# 3. Now ask about something totally unrelated to every document -- what happens?
print(open_book_answer("What is the boiling point of mercury?", my_docs))


Grounded in retrieved evidence: "The AIRTOMS railway delay-recovery model runs a two-stage Router plus Narrator LLM pipeline."
Grounded in retrieved evidence: "Acme Corp's customer churn rate improved from 6.2% to 4.1% after launching the loyalty program."


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

For the railway question, `open_book_answer` should correctly surface the new document, since it
shares words like "railway" and "delay" with the query.

For the mercury question, notice that `open_book_answer` **still returns a document** — it always
returns the *best available* match, even when nothing in the corpus is actually relevant. With word
overlap likely at zero for every document, `sorted()` just returns them in their original order,
and the function confidently reports the first one as "grounded evidence."

This is a real and important failure mode: naive retrieval has no concept of "nothing here is
relevant enough — say so instead." Chapter 10 covers explicit guardrails for this ("if the context
doesn't answer the question, say you don't know"), and Chapter 11 covers CRAG, which adds a
dedicated relevance evaluator specifically to catch cases like this before they reach generation.
</details>


## Chapter 3 — RAG Architecture Fundamentals

Every RAG system is really **two separate pipelines** that run at different times:

- **The offline indexing pipeline** — runs once (and periodically re-runs as documents change):
  load → clean → chunk → embed → store. This can take minutes to days, and happens entirely
  before any user query arrives.
- **The online query pipeline** — runs every single time a user asks a question, and must be
  fast: embed the query → search the index → (optionally rerank) → construct a prompt → generate.

Separating these two pipelines conceptually is the single most useful mental model for debugging a
RAG system: if answers are wrong, the first question is always *"is this a problem with what was
indexed, or a problem with what was retrieved and generated?"* — because the fixes are completely
different (re-chunk and re-embed the corpus, versus tune the retrieval or prompt logic).

### Modularity: RAG's superpower and its operational cost

Because indexing and querying are decoupled, every component in between — chunking strategy,
embedding model, vector store, retrieval algorithm, reranker, prompt template, generator model —
can be swapped independently. Below we define an empty pipeline skeleton with a placeholder method
for every stage. We'll fill in each method one at a time as we reach the matching chapter, in
exactly the order they're introduced conceptually.


In [7]:
# Chapter 3 -- A pipeline skeleton with every stage as a placeholder.
# We'll attach real implementations to each slot chapter by chapter.

class RAGPipeline:
    def __init__(self):
        self.chunker = None          # Chapter 4
        self.embedder = None         # Chapter 5
        self.index = None            # Chapter 6 (built by index_documents)
        self.retriever = None        # Chapters 6-8
        self.reranker = None         # Chapter 9 (optional)
        self.prompt_builder = None   # Chapter 10
        self.generator = None        # Chapter 7+ (a stand-in "LLM")

    def index_documents(self, documents):
        if self.chunker is None or self.embedder is None:
            raise NotImplementedError("Attach a chunker and embedder before indexing (see Ch.4-5).")
        chunks = self.chunker(documents)
        vectors = self.embedder(chunks)
        self.index = {"chunks": chunks, "vectors": vectors}
        return chunks

    def answer(self, query, k=3):
        if self.retriever is None:
            raise NotImplementedError("Attach a retriever before answering (see Ch.6-8).")
        candidates = self.retriever(query, self.index, k=k)
        if self.reranker is not None:
            candidates = self.reranker(query, candidates)
        if self.prompt_builder is None or self.generator is None:
            raise NotImplementedError("Attach a prompt_builder and generator (see Ch.7, Ch.10).")
        prompt = self.prompt_builder(query, candidates)
        return self.generator(prompt, candidates)

pipeline = RAGPipeline()
print("Empty RAGPipeline created. Attributes to fill in as we progress:")
for attr in ["chunker", "embedder", "retriever", "reranker", "prompt_builder", "generator"]:
    print(f"  - {attr}: {getattr(pipeline, attr)}")


Empty RAGPipeline created. Attributes to fill in as we progress:
  - chunker: None
  - embedder: None
  - retriever: None
  - reranker: None
  - prompt_builder: None
  - generator: None


Notice that calling `pipeline.answer(...)` right now raises a clear error rather than failing
silently — this is deliberate. As we progress through the notebook we'll attach a real component to
each slot, and by the end of Chapter 7 this exact `pipeline` object will be able to answer questions
end to end.


In [8]:
# Chapter 3 -- Confirming the pipeline fails loudly, not silently, when incomplete
try:
    pipeline.answer("What is RAG?")
except NotImplementedError as e:
    print("Expected error (pipeline is incomplete):", e)


Expected error (pipeline is incomplete): Attach a retriever before answering (see Ch.6-8).


### 🧪 Try it yourself

Without writing any new logic, attach a *trivial* generator to `pipeline.generator` that just
returns a fixed string, and a trivial `prompt_builder` that returns the query unchanged. Leave
`chunker`, `embedder`, `retriever` unset. What happens when you call `pipeline.answer(...)` now?
This should build your intuition for exactly which stage each error message is pointing at.


In [9]:
# TODO Your turn -- attach only generator and prompt_builder, leave the rest unset
pipeline.generator = lambda prompt, candidates: "This is a placeholder answer."
pipeline.prompt_builder = lambda query, candidates: query

try:
    pipeline.answer("What is RAG?")
except NotImplementedError as e:
    print("Error raised:", e)


Error raised: Attach a retriever before answering (see Ch.6-8).


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Even with `generator` and `prompt_builder` attached, `pipeline.answer(...)` still raises
`NotImplementedError`, but now for a *different* reason: `self.retriever is None`. The `answer`
method checks for a retriever **before** it ever reaches the prompt-builder or generator checks,
because retrieval happens first in the pipeline.

This is a useful debugging habit for real systems too: trace failures to the *earliest* stage in
the pipeline that's broken, not the symptom you happened to notice first. A missing retriever and a
bad generator can both cause "the answer is wrong," but the fixes are completely different.
</details>


## Chapter 4 — Document Loading & Chunking Strategies

Embedding models and LLM context windows both have practical size limits, and — more importantly
— retrieval works better on focused pieces of text than on whole documents. Chunking is the
process of splitting documents into smaller, semantically coherent pieces *before* embedding.

We'll implement three chunking strategies from scratch and compare their output on the same
document, so you can see exactly where each one draws its boundaries.

1. **Fixed-size chunking** — split every N characters, with overlap.
2. **Recursive character chunking** — prefer natural boundaries (paragraphs, then sentences),
   falling back to a hard cut only if nothing better is available.
3. **Structure-aware chunking** — respect the document's own headings/sections.

First, let's grow our corpus into something closer to a real (if small) multi-section document.


In [10]:
# Chapter 4 -- A small structured document to chunk (simulates a real report with headings)

SAMPLE_DOCUMENT = """# RAG Systems Overview

RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to
closed-book generation, because the model has real, traceable source text to draw from rather
than relying purely on its parametric memory.

## Chunking

Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant part. If chunks are too small,
context is lost -- a key fact and the sentence that explains it can end up in different chunks.

Structure-aware chunking respects headings and sections, which helps technical documents where
a table or a numbered list should not be split across chunk boundaries.

## Embeddings

Dense embeddings place semantically similar sentences close together in vector space, even when
the wording is completely different. This is what lets retrieval work by meaning instead of by
exact keyword match.

Query-document asymmetry means a question and its answer are rarely phrased the same way, which
is why some embedding models are trained with separate encoders for queries versus documents.
"""

print(SAMPLE_DOCUMENT[:300], "...")
print(f"\nTotal length: {len(SAMPLE_DOCUMENT)} characters")


# RAG Systems Overview

RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to
closed-book generation, because the model has real, traceable source text to draw from rather
than relying purely on its parametric memory.

## Chunking

Chunking determines what unit ...

Total length: 1142 characters


In [11]:
# Chapter 4 -- Strategy 1: Fixed-size chunking with overlap

def fixed_size_chunk(text, chunk_size=200, overlap=40):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

fixed_chunks = fixed_size_chunk(SAMPLE_DOCUMENT, chunk_size=200, overlap=40)
print(f"Fixed-size chunking produced {len(fixed_chunks)} chunks:\n")
for i, c in enumerate(fixed_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c[:120].replace(chr(10), " "), "...")
    print()


Fixed-size chunking produced 8 chunks:

--- chunk 0 (200 chars) ---
# RAG Systems Overview  RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to clo ...

--- chunk 1 (199 chars) ---
as real, traceable source text to draw from rather than relying purely on its parametric memory.  ## Chunking  Chunking  ...

--- chunk 2 (199 chars) ---
ved. If chunks are too large, precision drops because irrelevant text gets pulled in alongside the relevant part. If chu ...

--- chunk 3 (200 chars) ---
key fact and the sentence that explains it can end up in different chunks.  Structure-aware chunking respects headings a ...

--- chunk 4 (200 chars) ---
ents where a table or a numbered list should not be split across chunk boundaries.  ## Embeddings  Dense embeddings plac ...

--- chunk 5 (200 chars) ---
ogether in vector space, even when the wording is completely different. This is what lets retrieval work by meaning inst ...

--- chunk 6 (181 chars) ---
cument a

Notice that fixed-size chunking cuts through the middle of sentences and even words, wherever the
character count happens to land — it has zero awareness of the document's structure. This is fast
and simple, but it will happily split "hallucination" in half if that's where the boundary falls.

**Recursive character chunking** fixes this by trying a sequence of separators from "most natural"
to "least natural" — paragraph breaks first, then sentence boundaries, then finally a hard
character cut only as a last resort.


In [12]:
# Chapter 4 -- Strategy 2: Recursive character chunking
# Tries separators in order of "naturalness": paragraph break -> line break -> sentence -> word.

def recursive_chunk(text, chunk_size=200, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", ". ", " "]

    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []

    if not separators:
        # Last resort: hard character cut
        return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    sep = separators[0]
    remaining_seps = separators[1:]
    pieces = text.split(sep)

    chunks = []
    current = ""
    for piece in pieces:
        candidate = (current + sep + piece) if current else piece
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            if len(piece) > chunk_size:
                # This single piece is still too big -- recurse with the next separator down
                chunks.extend(recursive_chunk(piece, chunk_size, remaining_seps))
                current = ""
            else:
                current = piece
    if current:
        chunks.append(current.strip())

    return [c for c in chunks if c]

recursive_chunks = recursive_chunk(SAMPLE_DOCUMENT, chunk_size=200)
print(f"Recursive chunking produced {len(recursive_chunks)} chunks:\n")
for i, c in enumerate(recursive_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c[:150].replace(chr(10), " "), "...")
    print()


Recursive chunking produced 10 chunks:

--- chunk 0 (22 chars) ---
# RAG Systems Overview ...

--- chunk 1 (186 chars) ---
RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to closed-book generation, because the model has real, trace ...

--- chunk 2 (45 chars) ---
than relying purely on its parametric memory. ...

--- chunk 3 (11 chars) ---
## Chunking ...

--- chunk 4 (187 chars) ---
Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops because irrelevant text gets pulled in alongside the re ...

--- chunk 5 (95 chars) ---
context is lost -- a key fact and the sentence that explains it can end up in different chunks. ...

--- chunk 6 (181 chars) ---
Structure-aware chunking respects headings and sections, which helps technical documents where a table or a numbered list should not be split across c ...

--- chunk 7 (190 chars) ---
Dense embeddings place semantically similar sentences close together in vec

Recursive chunking already looks much better — chunks tend to end at sentence or paragraph
boundaries instead of mid-word. But it still doesn't know that our document has *headings* that
mark logical sections. **Structure-aware chunking** uses that structure directly: every chunk stays
within a single heading's section, and we record which heading it came from as metadata.


In [13]:
# Chapter 4 -- Strategy 3: Structure-aware chunking (respects Markdown headings)
# Each chunk carries metadata: which section (heading) it came from.

def structure_aware_chunk(text, chunk_size=250):
    lines = text.strip().split("\n")
    sections = []  # list of (heading, body_text)
    current_heading = "Untitled"
    current_body = []

    for line in lines:
        if line.startswith("#"):
            if current_body:
                sections.append((current_heading, "\n".join(current_body).strip()))
            current_heading = line.lstrip("#").strip()
            current_body = []
        else:
            current_body.append(line)
    if current_body:
        sections.append((current_heading, "\n".join(current_body).strip()))

    chunks = []
    for heading, body in sections:
        if not body:
            continue
        # Sub-chunk long sections using the recursive chunker, but tag every piece
        # with the heading it belongs to.
        for piece in recursive_chunk(body, chunk_size=chunk_size):
            chunks.append({"heading": heading, "text": piece})
    return chunks

structured_chunks = structure_aware_chunk(SAMPLE_DOCUMENT, chunk_size=250)
print(f"Structure-aware chunking produced {len(structured_chunks)} chunks:\n")
for i, c in enumerate(structured_chunks):
    print(f"--- chunk {i} | section: '{c['heading']}' ({len(c['text'])} chars) ---")
    print(c["text"][:150].replace(chr(10), " "), "...")
    print()


Structure-aware chunking produced 6 chunks:

--- chunk 0 | section: 'RAG Systems Overview' (232 chars) ---
RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to closed-book generation, because the model has real, trace ...

--- chunk 1 | section: 'Chunking' (187 chars) ---
Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops because irrelevant text gets pulled in alongside the re ...

--- chunk 2 | section: 'Chunking' (95 chars) ---
context is lost -- a key fact and the sentence that explains it can end up in different chunks. ...

--- chunk 3 | section: 'Chunking' (166 chars) ---
Structure-aware chunking respects headings and sections, which helps technical documents where a table or a numbered list should not be split across c ...

--- chunk 4 | section: 'Embeddings' (211 chars) ---
Dense embeddings place semantically similar sentences close together in vector space, even when the wording is completely d

Notice the `heading` field attached to every chunk. This is **metadata** — and it's what lets a
production system filter retrieval by section, reconstruct surrounding context after retrieval, and
give the generator enough information to produce a proper citation like *"According to the
Embeddings section..."* instead of just an unattributed quote.

> **⚠️ A concrete failure mode:** If a key fact spans two adjacent chunks — a number in one
> paragraph and its caveat in the next — no embedding model can recover that connection once the
> chunks are split and searched independently. This is one of the most common real-world causes of
> subtly wrong RAG answers, and is why generous overlap and/or hierarchical (parent-child) chunking
> matter in production.

> **🏭 Production note:** Real systems rarely write chunkers by hand. `langchain`'s
> `RecursiveCharacterTextSplitter` and `llama-index`'s node parsers implement the recursive and
> structure-aware strategies above (plus semantic chunking, which clusters by embedding similarity
> rather than character count) with far more edge-case handling — but the core algorithm is exactly
> what you just built.


### 🧪 Try it yourself

Try `chunk_size=80` with `recursive_chunk` on `SAMPLE_DOCUMENT`. Some sentences in our document are
longer than 80 characters — what does the chunker do when a single sentence can't fit within the
size limit even after trying every separator?


In [14]:
# TODO Your turn -- try a much smaller chunk_size and inspect what happens
small_chunks = recursive_chunk(SAMPLE_DOCUMENT, chunk_size=80)
print(f"Produced {len(small_chunks)} chunks with chunk_size=80\n")
for i, c in enumerate(small_chunks[:6]):
    print(f"[{i}] ({len(c)} chars) {c!r}")


Produced 26 chunks with chunk_size=80

[0] (22 chars) '# RAG Systems Overview'
[1] (52 chars) 'RAG systems ground LLM outputs in retrieved evidence'
[2] (38 chars) 'This reduces hallucination compared to'
[3] (76 chars) 'closed-book generation, because the model has real, traceable source text to'
[4] (16 chars) 'draw from rather'
[5] (45 chars) 'than relying purely on its parametric memory.'


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

With `chunk_size=80`, you'll see many more, much smaller chunks. When `recursive_chunk` runs out of
separators (it has tried `"\n\n"`, `"\n"`, `". "`, and finally `" "`) and a piece is *still* longer
than `chunk_size`, the function falls into its "last resort" branch: `separators` is empty, so it
does a hard character-by-character cut, exactly like `fixed_size_chunk` from earlier — potentially
splitting a word in half.

This is the real behavior of production recursive chunkers too: they try their best to respect
structure, but always have a fallback so no chunk silently exceeds the size limit. The lesson is
that `chunk_size` has a practical floor — set it too small relative to your sentence lengths, and
you lose the "natural boundary" benefit recursive chunking is supposed to give you.
</details>


## Chapter 5 — Embeddings & Vector Representations

An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text, such
that texts with similar meaning end up close together in that numerical space. This is what allows
retrieval to work by meaning rather than exact keyword match — a query about "vehicle won't start"
can retrieve a passage about "engine fails to turn over" even though they share almost no words.

Real production systems use neural embedding models (transformers trained specifically for this).
Those require downloading model weights, which we're avoiding in this notebook. Instead, we'll
build a **TF-IDF vectorizer from scratch** — a classical (pre-deep-learning) technique that still
produces genuine vector embeddings we can meaningfully compare. It's a legitimate embedding
method in its own right (and still used today, especially blended with dense methods), and every
mechanic — vectorizing text, comparing vectors, cosine similarity — transfers directly to real
embedding models.

### TF-IDF in one sentence

TF-IDF scores each word in a document by how often it appears *in that document* (term frequency)
weighted down by how common it is *across all documents* (inverse document frequency) — so common
words like "the" get suppressed, and distinctive words get amplified.


In [15]:
# Chapter 5 -- A TF-IDF "embedding" model, built entirely from scratch (no external API/model)

class TfidfEmbedder:
    def __init__(self):
        self.vocabulary = {}   # word -> index
        self.idf = None        # per-word inverse document frequency

    def _tokenize(self, text):
        return re.findall(r"[a-z0-9]+", text.lower())

    def fit(self, corpus):
        doc_freq = Counter()
        all_tokens = []
        for doc in corpus:
            tokens = set(self._tokenize(doc))
            all_tokens.append(tokens)
            doc_freq.update(tokens)

        self.vocabulary = {word: i for i, word in enumerate(sorted(doc_freq.keys()))}
        n_docs = len(corpus)
        self.idf = np.zeros(len(self.vocabulary))
        for word, idx in self.vocabulary.items():
            # Standard smoothed IDF: log(N / (1 + df)) + 1
            self.idf[idx] = math.log(n_docs / (1 + doc_freq[word])) + 1
        return self

    def transform(self, texts):
        vectors = np.zeros((len(texts), len(self.vocabulary)))
        for i, text in enumerate(texts):
            tokens = self._tokenize(text)
            tf = Counter(tokens)
            length = max(len(tokens), 1)
            for word, count in tf.items():
                if word in self.vocabulary:
                    idx = self.vocabulary[word]
                    vectors[i, idx] = (count / length) * self.idf[idx]
        return vectors

    def fit_transform(self, corpus):
        self.fit(corpus)
        return self.transform(corpus)

embedder = TfidfEmbedder()
doc_vectors = embedder.fit_transform(CORPUS_V1)

print(f"Vocabulary size: {len(embedder.vocabulary)} unique words")
print(f"Embedding matrix shape: {doc_vectors.shape}  (5 documents x {doc_vectors.shape[1]} dims)")
print(f"\nFirst document's vector (first 10 dims): {doc_vectors[0][:10].round(3)}")


Vocabulary size: 76 unique words
Embedding matrix shape: (5, 76)  (5 documents x 76 dims)

First document's vector (first 10 dims): [0.    0.    0.    0.    0.    0.128 0.    0.    0.    0.128]


### Comparing vectors: cosine similarity

Two embedding vectors are typically compared using **cosine similarity** — the cosine of the angle
between them, ranging from -1 (opposite) to 1 (identical direction), ignoring vector magnitude.
This is the standard choice for TF-IDF and most sentence embedding models.


In [16]:
# Chapter 5 -- Cosine similarity and our first real dense retriever

def cosine_similarity(vec_a, vec_b):
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))

def dense_retrieve(query, corpus, embedder, k=2):
    query_vec = embedder.transform([query])[0]
    corpus_vecs = embedder.transform(corpus)
    scores = [cosine_similarity(query_vec, doc_vec) for doc_vec in corpus_vecs]
    ranked = sorted(zip(scores, corpus), key=lambda x: x[0], reverse=True)
    return ranked[:k]

# The exact query that defeated naive_retrieve back in Chapter 1's exercise:
query = "old-fashioned keyword search"
results = dense_retrieve(query, CORPUS_V1, embedder, k=3)

print(f"QUERY: {query!r}\n")
for score, doc in results:
    print(f"  score={score:.4f}  {doc}")


QUERY: 'old-fashioned keyword search'

  score=0.2406  BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
  score=0.0000  RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
  score=0.0000  Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.


**Honest caveat:** TF-IDF is still fundamentally a word-overlap method — it counts shared terms,
just weighted more cleverly than raw counting. It will **not** solve the "old-fashioned keyword
search" vs. "sparse, keyword-based ranking" problem as well as a true neural embedding model would,
because it still has no notion of synonymy (it doesn't know "old-fashioned" relates to "sparse" in
this context). What TF-IDF *does* demonstrate correctly: partial word overlap ("keyword" appears in
both) now contributes a graded, weighted score instead of a blunt integer count, and rare/distinctive
words are weighted more heavily than common ones. A real transformer embedding model (below) closes
the remaining semantic gap.

> **🏭 Production note — self-hosted vs. API embeddings:**
>
> | | Self-hosted (BGE, GTE-Qwen2) | API-based (OpenAI, Cohere) |
> |---|---|---|
> | Cost | One-time compute cost, free per-query after | Pay per token, ongoing |
> | Privacy | Data never leaves your infrastructure | Data sent to a third party |
> | Setup | Requires hosting the model | Single API call |
>
> To swap in a real model, replace `TfidfEmbedder` with, for example:
> ```python
> from sentence_transformers import SentenceTransformer
> model = SentenceTransformer("BAAI/bge-small-en-v1.5")
> doc_vectors = model.encode(CORPUS_V1, normalize_embeddings=True)
> ```
> Everything downstream — `cosine_similarity`, `dense_retrieve` — works unchanged, because the
> *interface* (text in, vector out) is identical. This is the modularity payoff from Chapter 3.

> **⚠️ Normalization pitfall:** Many neural embedding models are trained assuming their output
> vectors will be L2-normalized before comparison. Using raw dot product on un-normalized vectors
> from such a model silently produces a systematically worse ranking. Always check your specific
> model's documentation for its intended distance metric.


### 🧪 Try it yourself

`cosine_similarity` ignores vector magnitude entirely — it only cares about direction. Write a
quick test: take any document vector, multiply it by 100 (`doc_vectors[0] * 100`), and confirm its
cosine similarity to the *original* (unscaled) vector is still 1.0. This is exactly why cosine
similarity is a popular default: it isn't thrown off by documents of very different lengths having
very different vector magnitudes.


In [17]:
# TODO Your turn -- confirm cosine similarity is scale-invariant
original = doc_vectors[0]
scaled = doc_vectors[0] * 100

similarity = cosine_similarity(original, scaled)
print(f"Cosine similarity between a vector and a 100x-scaled copy of itself: {similarity:.6f}")

# Compare: what does plain dot product give instead?
dot_product = float(np.dot(original, scaled))
print(f"Plain dot product (NOT scale-invariant): {dot_product:.6f}")


Cosine similarity between a vector and a 100x-scaled copy of itself: 1.000000
Plain dot product (NOT scale-invariant): 23.245956


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Cosine similarity should print almost exactly `1.000000` — scaling a vector doesn't change its
*direction*, only its length, and cosine similarity only measures the angle between vectors.

Plain dot product, on the other hand, is **not** scale-invariant — multiplying one vector by 100
multiplies the dot product by roughly 100 as well. This is exactly the pitfall mentioned above: if
you accidentally use dot product where a model expects cosine similarity (or vice versa, on
un-normalized vectors), you'll get a systematically distorted ranking that still "looks" plausible,
which makes it a dangerous silent bug rather than a loud crash.
</details>


## Chapter 6 — Vector Databases & Indexing

Once documents are chunked and embedded, the naive way to find the best match for a query is
**brute-force search**: compare the query vector against every single stored vector. This is exact
and simple, but its cost grows linearly with corpus size — fine for 5 documents, far too slow for a
million-chunk corpus in an interactive application.

Real vector databases (FAISS, Chroma, Qdrant, Pinecone, Weaviate) use **Approximate Nearest
Neighbour (ANN)** indexing structures — commonly HNSW (a layered navigable graph) or IVF
(cluster-based partitioning) — trading a small amount of accuracy for a large amount of speed.

We'll build a clean, general-purpose `VectorIndex` class using brute-force search as a clear stand-in
for a real ANN index. The *interface* — `add`, `search` — is exactly what a real vector database
exposes; only the internal search algorithm differs.


In [18]:
# Chapter 6 -- A minimal in-memory vector index (brute-force ANN stand-in)
# Interface mirrors a real vector database: add(), search(). Only the internals differ.

class VectorIndex:
    def __init__(self, embedder):
        self.embedder = embedder
        self.vectors = None      # np.ndarray, shape (n_docs, n_dims)
        self.texts = []          # original chunk text
        self.metadata = []       # parallel list of metadata dicts

    def add(self, texts, metadata=None):
        if metadata is None:
            metadata = [{} for _ in texts]
        new_vectors = self.embedder.transform(texts)
        if self.vectors is None:
            self.vectors = new_vectors
        else:
            self.vectors = np.vstack([self.vectors, new_vectors])
        self.texts.extend(texts)
        self.metadata.extend(metadata)
        return len(texts)

    def search(self, query, k=3):
        if self.vectors is None or len(self.texts) == 0:
            return []
        query_vec = self.embedder.transform([query])[0]
        # Brute-force: compute cosine similarity against every stored vector.
        norms = np.linalg.norm(self.vectors, axis=1)
        query_norm = np.linalg.norm(query_vec)
        denom = np.where(norms * query_norm == 0, 1e-10, norms * query_norm)
        scores = (self.vectors @ query_vec) / denom

        top_k_idx = np.argsort(-scores)[:k]
        return [
            {"text": self.texts[i], "score": float(scores[i]), "metadata": self.metadata[i]}
            for i in top_k_idx
        ]

    def __len__(self):
        return len(self.texts)


# Build and populate an index from our Chapter 4 structured chunks
embedder_v2 = TfidfEmbedder()
embedder_v2.fit([c["text"] for c in structured_chunks])

index = VectorIndex(embedder_v2)
index.add(
    texts=[c["text"] for c in structured_chunks],
    metadata=[{"heading": c["heading"]} for c in structured_chunks],
)

print(f"Index built with {len(index)} chunks.")
results = index.search("how does structure-aware chunking help technical documents", k=2)
for r in results:
    print(f"\nscore={r['score']:.4f}  section='{r['metadata']['heading']}'")
    print(" ", r["text"][:140].replace(chr(10), " "), "...")


Index built with 6 chunks.

score=0.4307  section='Chunking'
  Structure-aware chunking respects headings and sections, which helps technical documents where a table or a numbered list should not be spli ...

score=0.0603  section='Embeddings'
  Query-document asymmetry means a question and its answer are rarely phrased the same way, which is why some embedding models are trained wit ...


### Why brute-force doesn't scale (and what replaces it)

Let's make the scaling problem concrete: brute-force search does `O(n)` vector comparisons per
query. Below, we simulate the *relative* cost of brute-force search as corpus size grows, to build
intuition for why real systems need ANN indexing at scale.

| Index type | How it works | Trade-off |
|---|---|---|
| Brute-force | Compare against every vector | Exact, but O(n) per query |
| **HNSW** | Multi-layer graph; navigate coarse-to-fine | Very fast, small accuracy loss, handles updates well |
| **IVF** | Cluster vectors ahead of time; search only nearby clusters | Fast, needs periodic re-clustering |


In [19]:
# Chapter 6 -- Simulating how brute-force search cost scales with corpus size
import time

def time_brute_force_search(n_docs, n_dims=50, n_queries=20):
    rng = np.random.default_rng(0)
    vectors = rng.random((n_docs, n_dims))
    queries = rng.random((n_queries, n_dims))

    start = time.perf_counter()
    for q in queries:
        norms = np.linalg.norm(vectors, axis=1)
        scores = (vectors @ q) / (norms * np.linalg.norm(q) + 1e-10)
        top_k = np.argsort(-scores)[:5]
    elapsed = time.perf_counter() - start
    return elapsed

print(f"{'Corpus size':>12} | {'Time for 20 queries':>20}")
print("-" * 37)
for n in [1_000, 10_000, 100_000]:
    elapsed = time_brute_force_search(n)
    print(f"{n:>12,} | {elapsed*1000:>17.2f} ms")


 Corpus size |  Time for 20 queries
-------------------------------------
       1,000 |              5.61 ms
      10,000 |             37.70 ms
     100,000 |            513.92 ms


You should see search time grow roughly linearly with corpus size — exactly the `O(n)` behavior
that motivates ANN indexing at real scale (millions of chunks, sub-100ms latency requirements).

> **🏭 Production note:** Swapping `VectorIndex` for a real ANN-backed store is a matter of
> changing the storage/search backend, not the pipeline around it:
> ```python
> import faiss
> index = faiss.IndexHNSWFlat(dims, 32)   # HNSW with 32 neighbors per node
> index.add(doc_vectors)
> distances, indices = index.search(query_vector.reshape(1, -1), k=5)
> ```
> For a system already using PostgreSQL, **pgvector** is often the pragmatic first choice — it adds
> vector search to your existing database instead of standing up a separate system.


### 🧪 Try it yourself

`VectorIndex.add()` can be called multiple times to add documents incrementally, without rebuilding
the whole index. Prove this by adding a brand-new chunk to our existing `index`, and searching for
it specifically. Then check `len(index)` to confirm the old chunks are still there too.


In [20]:
# TODO Your turn -- add a new chunk incrementally and search for it
before_count = len(index)

index.add(
    texts=["Reranking uses a cross-encoder to re-score a short candidate list after initial retrieval."],
    metadata=[{"heading": "Reranking"}],
)

print(f"Index size before: {before_count}, after: {len(index)}")

results = index.search("what re-scores candidates after retrieval", k=1)
print("\nTop result:", results[0]["text"])
print("From section:", results[0]["metadata"]["heading"])


Index size before: 6, after: 7

Top result: Reranking uses a cross-encoder to re-score a short candidate list after initial retrieval.
From section: Reranking


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

The index size grows by exactly 1 after adding the new chunk, and the old chunks are untouched —
`np.vstack` appends the new vector onto the existing matrix rather than replacing it. The search
for "what re-scores candidates after retrieval" should correctly surface the new reranking chunk,
since it shares strong vocabulary overlap ("re-scores"/"re-score", "candidates"/"candidate").

This incremental-add capability matters a lot in production: it means you don't have to rebuild an
entire multi-million-chunk index every time one new document arrives. Real ANN indexes like HNSW
support incremental inserts too, though with some caveats — very frequent inserts/deletes can
degrade an HNSW graph's search quality over time, which is why production systems often schedule
periodic full re-indexing even when incremental updates are supported.
</details>


## Chapter 7 — Building Your First End-to-End Naive RAG Pipeline

Time to connect everything: chunk the documents → embed each chunk → store in the index → embed
the query → search for the top-k chunks → build a prompt → generate an answer. This is commonly
called **"Naive RAG"** — not an insult, but the honest baseline every more sophisticated technique
in this notebook is measured against.

We'll fill in the empty `RAGPipeline` skeleton from Chapter 3 with the real components we've built
in Chapters 4–6, plus a simple stand-in "generator" (a real LLM call arrives conceptually in
Chapter 10, when we build a proper grounded prompt template).


In [21]:
# Chapter 7 -- Assembling the full naive RAG pipeline from Chapters 4-6

def make_naive_pipeline(documents):
    # 1. Chunk (Chapter 4's structure-aware chunker)
    all_chunks = []
    for doc in documents:
        all_chunks.extend(structure_aware_chunk(doc, chunk_size=250))

    # 2. Embed + 3. Index (Chapter 5's TF-IDF embedder + Chapter 6's VectorIndex)
    texts = [c["text"] for c in all_chunks]
    metas = [{"heading": c["heading"]} for c in all_chunks]

    pipeline_embedder = TfidfEmbedder()
    pipeline_embedder.fit(texts)

    pipeline_index = VectorIndex(pipeline_embedder)
    pipeline_index.add(texts, metas)

    return pipeline_index

def naive_rag_answer(query, index, k=2):
    # 4. Retrieve
    retrieved = index.search(query, k=k)

    # 5. Build a (very simple) prompt -- a proper grounded template arrives in Chapter 10
    context = "\n".join(f"- {r['text']}" for r in retrieved)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer using only the context above:"

    # 6. Generate -- stand-in "LLM": just surfaces the single best-matching sentence.
    #    A real system replaces this with an actual LLM API call using the prompt above.
    if not retrieved or retrieved[0]["score"] < 0.25:
        answer = "I don't have enough relevant information to answer this."
    else:
        answer = f"Based on the retrieved context: \"{retrieved[0]['text']}\""

    return {"query": query, "retrieved": retrieved, "prompt": prompt, "answer": answer}


naive_index = make_naive_pipeline([SAMPLE_DOCUMENT])
print(f"Indexed {len(naive_index)} chunks from SAMPLE_DOCUMENT.\n")

result = naive_rag_answer("What problems does chunking cause if chunks are too large?", naive_index, k=2)
print("QUERY: ", result["query"])
print("ANSWER:", result["answer"])


Indexed 6 chunks from SAMPLE_DOCUMENT.

QUERY:  What problems does chunking cause if chunks are too large?
ANSWER: Based on the retrieved context: "Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant part. If chunks are too small,"


Let's look at the full constructed prompt — this is exactly what would be sent to a real LLM API in
a production system.


In [22]:
# Chapter 7 -- Inspecting the full constructed prompt
print(result["prompt"])


Context:
- Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant part. If chunks are too small,
- Query-document asymmetry means a question and its answer are rarely phrased the same way, which
is why some embedding models are trained with separate encoders for queries versus documents.

Question: What problems does chunking cause if chunks are too large?
Answer using only the context above:


### Where Naive RAG breaks down

Naive RAG works reasonably well for simple, single-fact questions over a clean corpus. Let's
deliberately break it with a question type it's known to struggle with — one requiring information
that isn't phrased the way it's asked about.


In [23]:
# Chapter 7 -- Deliberately probing a known weak point: vocabulary mismatch
tricky_query = "why shouldn't retrieval units be too big"  # means the same as "chunks too large"

result2 = naive_rag_answer(tricky_query, naive_index, k=2)
print("QUERY: ", result2["query"])
print("TOP RETRIEVED SCORE:", round(result2["retrieved"][0]["score"], 4))
print("ANSWER:", result2["answer"])


QUERY:  why shouldn't retrieval units be too big
TOP RETRIEVED SCORE: 0.17
ANSWER: I don't have enough relevant information to answer this.


Depending on the exact score, you should see the retrieval score drop noticeably compared to the
first query — TF-IDF still leans on shared vocabulary, and "why shouldn't retrieval units be too
big" barely overlaps with "if chunks are too large, precision drops." This is exactly the
**vocabulary mismatch** failure mode discussed in the reference document, and it's the main reason
production systems layer hybrid search (Chapter 8) and query rewriting (Chapter 9) on top of Naive
RAG rather than shipping it as-is.

**Naive RAG's other known weaknesses**, each motivating a later chapter:
- **No quality control on retrieved chunks** — top-k similarity doesn't guarantee genuine
  relevance (→ Chapter 9 reranking, Chapter 11 CRAG).
- **Multi-hop questions fail** — connecting facts across multiple documents is invisible to
  single-pass similarity search (→ Chapter 12 GraphRAG, Chapter 13 agentic RAG).
- **No self-awareness** — it retrieves and generates every time, even when nothing relevant was
  found (→ Chapter 11 Self-RAG / Adaptive-RAG).

> **🏭 Production note:** Every production RAG system in the world is, underneath its added
> sophistication, still running this same core loop. Understanding Naive RAG's failure modes is
> what makes every later chapter feel like a targeted fix rather than an arbitrary new technique.


### 🧪 Try it yourself

The threshold `retrieved[0]["score"] < 0.25` in `naive_rag_answer` decides when to say "I don't
have enough information" instead of confidently (and possibly wrongly) answering. Ask about
something totally absent from `SAMPLE_DOCUMENT` — e.g. "What is the boiling point of mercury?" —
and check the top retrieval score. Is it low enough to trigger the "I don't know" branch? If not,
what does that tell you about relying on a single raw similarity score as a relevance guardrail?


In [24]:
# TODO Your turn -- test the "I don't know" threshold with an off-topic question
off_topic = naive_rag_answer("What is the boiling point of mercury?", naive_index, k=2)
print("TOP SCORE:", round(off_topic["retrieved"][0]["score"], 4))
print("ANSWER:   ", off_topic["answer"])

# Surprising? The score probably ISN'T near zero, even though mercury is unrelated to our corpus.
# Read the solution below for why.


TOP SCORE: 0.3101
ANSWER:    Based on the retrieved context: "Dense embeddings place semantically similar sentences close together in vector space, even when
the wording is completely different. This is what lets retrieval work by meaning instead of by
exact keyword match."


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Surprisingly, the "boiling point of mercury" query probably does **not** score near zero, even
though our corpus has nothing to do with chemistry. This is a genuine and important limitation of
TF-IDF cosine similarity on a *tiny* corpus: with only a handful of chunks and a small shared
vocabulary (common words like "is," "the," "of" still contribute some weight even after IDF
down-weighting), almost every query ends up with *some* nonzero similarity to *something* — there's
no natural "zero" to fall back on the way there might be with a much larger, more diverse corpus.

This is exactly the failure mode flagged in the reference document's Chapter 2 discussion: naive
retrieval has no built-in concept of "nothing here is relevant enough — say so instead." A single
raw similarity threshold is a fragile guardrail, because what counts as "low" depends heavily on
corpus size and diversity, and can't be picked once and trusted everywhere.

Two real fixes, both introduced later in this notebook:
- **Chapter 9** — reranking with a cross-encoder gives a more calibrated, query-aware relevance
  signal than raw cosine similarity.
- **Chapter 11** — CRAG's dedicated relevance evaluator explicitly classifies each retrieved
  document as Correct / Ambiguous / Incorrect, rather than trusting one threshold on one score.
</details>


## Chapter 8 — Retrieval Techniques: Sparse, Dense & Hybrid Search

Before embeddings existed, information retrieval was dominated by sparse, keyword-based methods —
and the most important of these, **BM25**, is still widely used today, often *alongside* dense
retrieval rather than instead of it.

- **Sparse (BM25):** scores documents by exact term overlap, weighted by how rare each term is
  across the corpus, adjusted for document length. Unbeatable for exact-match queries (product
  codes, names, acronyms).
- **Dense (embeddings):** captures meaning even when wording differs, as we saw in Chapter 5.

Because they fail in different, largely non-overlapping ways, combining them — **hybrid search** —
consistently outperforms either alone. We'll implement BM25 from scratch, then fuse it with our
Chapter 5 dense scores using **Reciprocal Rank Fusion (RRF)**.


In [25]:
# Chapter 8 -- BM25 (sparse retrieval), implemented from scratch

class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1   # controls term-frequency saturation
        self.b = b     # controls document-length normalization strength
        self.corpus_tokens = []
        self.doc_lens = []
        self.avg_doc_len = 0
        self.doc_freq = Counter()   # how many documents each term appears in
        self.n_docs = 0

    def _tokenize(self, text):
        return re.findall(r"[a-z0-9]+", text.lower())

    def fit(self, corpus):
        self.corpus_tokens = [self._tokenize(d) for d in corpus]
        self.doc_lens = [len(t) for t in self.corpus_tokens]
        self.avg_doc_len = sum(self.doc_lens) / len(self.doc_lens)
        self.n_docs = len(corpus)
        self.doc_freq = Counter()
        for tokens in self.corpus_tokens:
            for term in set(tokens):
                self.doc_freq[term] += 1
        return self

    def _idf(self, term):
        # Standard BM25 IDF with a floor so common terms don't go negative.
        n_containing = self.doc_freq.get(term, 0)
        return max(math.log((self.n_docs - n_containing + 0.5) / (n_containing + 0.5) + 1), 0.0)

    def score(self, query, doc_index):
        query_terms = self._tokenize(query)
        doc_tokens = self.corpus_tokens[doc_index]
        doc_term_counts = Counter(doc_tokens)
        doc_len = self.doc_lens[doc_index]

        total = 0.0
        for term in query_terms:
            if term not in doc_term_counts:
                continue
            freq = doc_term_counts[term]
            idf = self._idf(term)
            numerator = freq * (self.k1 + 1)
            denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_len)
            total += idf * (numerator / denominator)
        return total

    def search(self, query, k=5):
        scores = [(self.score(query, i), i) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:k]


bm25 = BM25()
bm25.fit(CORPUS_V1)

query = "keyword ranking function"
bm25_results = bm25.search(query, k=5)

print(f"BM25 results for {query!r}:\n")
for score, idx in bm25_results:
    print(f"  score={score:.4f}  {CORPUS_V1[idx]}")


BM25 results for 'keyword ranking function':

  score=3.9936  BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
  score=0.0000  RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
  score=0.0000  Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.
  score=0.0000  Dense embeddings place semantically similar sentences close together in vector space, even with different wording.
  score=0.0000  A reranker takes the shortlist from initial retrieval and re-scores it with a more accurate but slower model.


Now let's compare BM25 against dense retrieval (our Chapter 5 TF-IDF embedder) on the exact same
query, side by side.


In [26]:
# Chapter 8 -- Sparse (BM25) vs. Dense (TF-IDF embeddings) on the same query, side by side

query = "old-fashioned keyword search"

bm25_ranked = bm25.search(query, k=5)
dense_ranked = dense_retrieve(query, CORPUS_V1, embedder, k=5)

print(f"QUERY: {query!r}\n")
print(f"{'Rank':<5}{'BM25 (sparse)':<70}{'Dense (embeddings)'}")
print("-" * 140)
for rank in range(5):
    bm25_doc = CORPUS_V1[bm25_ranked[rank][1]][:55] + "..."
    dense_doc = dense_ranked[rank][1][:55] + "..."
    print(f"{rank+1:<5}{bm25_doc:<70}{dense_doc}")


QUERY: 'old-fashioned keyword search'

Rank BM25 (sparse)                                                         Dense (embeddings)
--------------------------------------------------------------------------------------------------------------------------------------------
1    BM25 is a sparse, keyword-based ranking function that p...            BM25 is a sparse, keyword-based ranking function that p...
2    RAG systems ground LLM outputs in retrieved evidence, r...            RAG systems ground LLM outputs in retrieved evidence, r...
3    Chunking determines what unit of text gets retrieved; t...            Chunking determines what unit of text gets retrieved; t...
4    Dense embeddings place semantically similar sentences c...            Dense embeddings place semantically similar sentences c...
5    A reranker takes the shortlist from initial retrieval a...            A reranker takes the shortlist from initial retrieval a...


### Reciprocal Rank Fusion (RRF)

The two rankings above likely disagree, at least partially — that's expected, since sparse and
dense retrieval fail differently. **RRF** combines multiple ranked lists into one fused ranking
using a simple, training-free formula: for each document, sum `1 / (k + rank)` across every list it
appears in, where `k` is a constant (commonly 60) and `rank` is its position (1-indexed) in that
list. A document that ranks highly in *both* lists gets a high fused score — even though BM25 and
cosine similarity scores are on completely different numeric scales, since RRF only looks at rank,
never the raw score.


In [27]:
# Chapter 8 -- Reciprocal Rank Fusion: merging BM25 and dense rankings into one hybrid ranking

def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    ranked_lists: a list of ranked lists, each a list of document texts in rank order
    (best first). Returns a single fused ranking as (fused_score, doc_text) tuples.
    """
    fused_scores = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            fused_scores[doc] += 1.0 / (k + rank)
    fused = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [(score, doc) for doc, score in fused]

def hybrid_search(query, corpus, bm25_index, embedder, k=5):
    bm25_ranked = [corpus[i] for _, i in bm25_index.search(query, k=len(corpus))]
    dense_ranked = [doc for _, doc in dense_retrieve(query, corpus, embedder, k=len(corpus))]
    fused = reciprocal_rank_fusion([bm25_ranked, dense_ranked])
    return fused[:k]

query = "old-fashioned keyword search"
hybrid_results = hybrid_search(query, CORPUS_V1, bm25, embedder, k=5)

print(f"Hybrid (RRF-fused) ranking for {query!r}:\n")
for score, doc in hybrid_results:
    print(f"  fused_score={score:.4f}  {doc}")


Hybrid (RRF-fused) ranking for 'old-fashioned keyword search':

  fused_score=0.0328  BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
  fused_score=0.0323  RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
  fused_score=0.0317  Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.
  fused_score=0.0312  Dense embeddings place semantically similar sentences close together in vector space, even with different wording.
  fused_score=0.0308  A reranker takes the shortlist from initial retrieval and re-scores it with a more accurate but slower model.


> **📊 Reported impact:** Multiple 2025–2026 production retrieval studies report hybrid search
> improving recall by roughly 1–17% over sparse-only or dense-only retrieval alone, and industry
> surveys report a substantial majority of production RAG deployments — commonly cited around 72%
> — now use some form of hybrid retrieval.

> **🏭 Production note:** Real vector databases increasingly support hybrid search natively —
> Qdrant, Weaviate, and Elasticsearch/OpenSearch all expose combined sparse+dense queries with
> built-in RRF (or similar) fusion, so you typically won't hand-roll `reciprocal_rank_fusion` in
> production, but the underlying logic is exactly what you just implemented.


### 🧪 Try it yourself

Try `query = "BM25 sparse ranking"` (a query that shares exact vocabulary with `CORPUS_V1[3]`) with
`hybrid_search`. Compare its fused ranking to what `bm25.search` alone returns for the same query.
Do they agree on the #1 result? Now try a query where BM25 and dense retrieval strongly *disagree*
on the #1 result (hint: revisit the side-by-side table above) — what does RRF do when the two
input rankings conflict?


In [28]:
# TODO Your turn -- compare hybrid vs. BM25-only on a keyword-heavy query
exact_query = "BM25 sparse ranking"

print("BM25 alone:")
for score, idx in bm25.search(exact_query, k=3):
    print(f"  score={score:.4f}  {CORPUS_V1[idx]}")

print("\nHybrid (RRF):")
for score, doc in hybrid_search(exact_query, CORPUS_V1, bm25, embedder, k=3):
    print(f"  score={score:.4f}  {doc}")


BM25 alone:
  score=3.9936  BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
  score=0.0000  RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
  score=0.0000  Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.

Hybrid (RRF):
  score=0.0328  BM25 is a sparse, keyword-based ranking function that predates embeddings and still outperforms them on exact-term queries.
  score=0.0323  RAG systems ground LLM outputs in retrieved evidence, reducing hallucination compared to closed-book generation.
  score=0.0317  Chunking determines what unit of text gets retrieved; too large and precision drops, too small and context is lost.


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

For `"BM25 sparse ranking"`, both BM25-alone and the hybrid ranking should agree on the same top
result — the BM25 passage (`CORPUS_V1[3]`) — since it shares exact, distinctive vocabulary with the
query and BM25 will rank it strongly, which RRF then reinforces.

When the two input rankings genuinely disagree on which document is best, RRF doesn't pick a
"winner" between them — it rewards documents that rank *reasonably well in both* lists over
documents that rank #1 in one list but poorly in the other. A document ranked #1 by BM25 but #5 by
dense retrieval gets `1/(60+1) + 1/(60+5) ≈ 0.0318`, while a document ranked #2 in both gets
`1/(60+2) + 1/(60+2) ≈ 0.0323` — slightly *higher*, because RRF favors consistent, moderate
agreement over a single method's strong top pick. This "hedge against either method being wrong"
behavior is exactly why hybrid search tends to be more robust than either method alone.
</details>


## Chapter 9 — Query Transformation & Reranking

A user's literal question is often a poor match for how the answer is actually phrased in the
source documents. **Query transformation** techniques rewrite or expand the query before
retrieval; **reranking** adds a second, more accurate pass over the initial shortlist.

- **HyDE (Hypothetical Document Embeddings):** instead of embedding the question directly, an LLM
  first generates a plausible *hypothetical answer*, and that answer's embedding is used to search
  — because a hypothetical answer is written in the same style as a real answer, making it a
  better match target than the question itself.
- **Cross-encoder reranking:** a bi-encoder (used for initial retrieval) embeds query and document
  *separately* and compares vectors — fast, but limited. A cross-encoder feeds the query and
  document *together* into one model, letting it directly attend to word-level matches — slower,
  but more accurate. Used only on the small shortlist retrieval already returned.

We don't have a real LLM available in this notebook, so we'll simulate both techniques locally:
HyDE with a lookup-based "hypothetical answer generator," and the reranker with a lexical-overlap
heuristic that behaves like a (much simpler) cross-encoder.


In [29]:
# Chapter 9 -- A toy, locally-simulated HyDE generator (no real LLM call)
# A real system would prompt an LLM: "Write a passage that answers this question."
# We simulate that by hand-writing a few hypothetical answers keyed by topic keywords.

HYPOTHETICAL_ANSWERS = {
    "keyword": "Sparse keyword-based ranking functions like BM25 score documents by exact term overlap, weighted by term rarity across the corpus.",
    "meaning": "Dense embedding models capture semantic meaning, placing similar concepts near each other in vector space regardless of exact wording.",
    "rescore": "A reranker re-scores a shortlist of retrieved candidates using a more accurate, slower model than the one used for initial retrieval.",
}

def hyde_generate(query):
    # Toy "LLM": picks the closest matching hypothetical answer template by keyword.
    q = query.lower()
    if any(w in q for w in ["keyword", "sparse", "exact"]):
        return HYPOTHETICAL_ANSWERS["keyword"]
    if any(w in q for w in ["meaning", "semantic", "similar"]):
        return HYPOTHETICAL_ANSWERS["meaning"]
    if any(w in q for w in ["rescore", "rerank", "shortlist"]):
        return HYPOTHETICAL_ANSWERS["rescore"]
    return query  # fallback: no good hypothetical answer, just use the query as-is

def hyde_retrieve(query, corpus, embedder, k=3):
    hypothetical = hyde_generate(query)
    # Embed the HYPOTHETICAL ANSWER, not the raw query -- this is the core HyDE trick.
    return dense_retrieve(hypothetical, corpus, embedder, k=k), hypothetical

query = "old-fashioned keyword search"
plain_results = dense_retrieve(query, CORPUS_V1, embedder, k=3)
hyde_results, hypothetical_used = hyde_retrieve(query, CORPUS_V1, embedder, k=3)

print(f"QUERY: {query!r}")
print(f"HYDE'S HYPOTHETICAL ANSWER: {hypothetical_used!r}\n")

print("Plain dense retrieval (embeds the query directly):")
for score, doc in plain_results:
    print(f"  score={score:.4f}  {doc[:70]}...")

print("\nHyDE retrieval (embeds the hypothetical answer):")
for score, doc in hyde_results:
    print(f"  score={score:.4f}  {doc[:70]}...")


QUERY: 'old-fashioned keyword search'
HYDE'S HYPOTHETICAL ANSWER: 'Sparse keyword-based ranking functions like BM25 score documents by exact term overlap, weighted by term rarity across the corpus.'

Plain dense retrieval (embeds the query directly):
  score=0.2406  BM25 is a sparse, keyword-based ranking function that predates embeddi...
  score=0.0000  RAG systems ground LLM outputs in retrieved evidence, reducing halluci...
  score=0.0000  Chunking determines what unit of text gets retrieved; too large and pr...

HyDE retrieval (embeds the hypothetical answer):
  score=0.5804  BM25 is a sparse, keyword-based ranking function that predates embeddi...
  score=0.0701  A reranker takes the shortlist from initial retrieval and re-scores it...
  score=0.0000  RAG systems ground LLM outputs in retrieved evidence, reducing halluci...


### A toy cross-encoder reranker

A real cross-encoder is a transformer that takes `(query, document)` as one joint input and outputs
a relevance score — it can directly notice things like "this document contains the exact phrase
from the query" in a way that comparing two independently-computed vectors cannot. We'll simulate
the *behavior* (not the neural architecture) with a richer scoring function that looks at more
signals than plain cosine similarity: exact substring matches, word order, and word overlap
combined.


In [30]:
# Chapter 9 -- A toy cross-encoder-style reranker
# Real cross-encoders use a transformer; we simulate the BEHAVIOR (joint query-doc scoring
# with more signal than cosine similarity alone) using simple lexical heuristics.

def toy_cross_encoder_score(query, document):
    q_words = query.lower().split()
    d_words = document.lower().split()
    q_set, d_set = set(q_words), set(d_words)

    overlap_score = len(q_set & d_set) / max(len(q_set), 1)

    # Bonus: exact substring match (something cosine similarity on pooled vectors can miss)
    substring_bonus = 0.3 if query.lower() in document.lower() else 0.0

    # Bonus: shared bigrams reward matching word ORDER, not just word presence
    q_bigrams = set(zip(q_words, q_words[1:]))
    d_bigrams = set(zip(d_words, d_words[1:]))
    bigram_bonus = 0.2 * (len(q_bigrams & d_bigrams) / max(len(q_bigrams), 1))

    return overlap_score + substring_bonus + bigram_bonus

def rerank(query, candidates, top_n=None):
    """candidates: list of document texts (already retrieved). Returns re-scored, re-sorted list."""
    scored = [(toy_cross_encoder_score(query, doc), doc) for doc in candidates]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_n] if top_n else scored


# Take a shortlist from initial (dense) retrieval, then rerank it
query = "reciprocal rank fusion combining sparse and dense"
initial_shortlist = [doc for _, doc in dense_retrieve(query, CORPUS_V1, embedder, k=5)]

print("Initial retrieval order:")
for i, doc in enumerate(initial_shortlist):
    print(f"  {i+1}. {doc[:70]}...")

reranked = rerank(query, initial_shortlist)
print("\nAfter reranking:")
for i, (score, doc) in enumerate(reranked):
    print(f"  {i+1}. score={score:.3f}  {doc[:70]}...")


Initial retrieval order:
  1. BM25 is a sparse, keyword-based ranking function that predates embeddi...
  2. Dense embeddings place semantically similar sentences close together i...
  3. Chunking determines what unit of text gets retrieved; too large and pr...
  4. A reranker takes the shortlist from initial retrieval and re-scores it...
  5. RAG systems ground LLM outputs in retrieved evidence, reducing halluci...

After reranking:
  1. score=0.143  BM25 is a sparse, keyword-based ranking function that predates embeddi...
  2. score=0.143  Dense embeddings place semantically similar sentences close together i...
  3. score=0.143  Chunking determines what unit of text gets retrieved; too large and pr...
  4. score=0.143  A reranker takes the shortlist from initial retrieval and re-scores it...
  5. score=0.000  RAG systems ground LLM outputs in retrieved evidence, reducing halluci...


> **🏭 Production note:** Real cross-encoder rerankers include Cohere's `rerank-v3` (API) and the
> BGE-reranker family (self-hosted, open-weight). The interface is nearly identical to `rerank()`
> above:
> ```python
> from sentence_transformers import CrossEncoder
> model = CrossEncoder("BAAI/bge-reranker-base")
> scores = model.predict([(query, doc) for doc in candidates])
> ```
> The key architectural rule stands regardless of implementation: cross-encoders are too slow to
> run over an entire corpus, so they're only ever applied to the small shortlist (typically top
> 20–50) that a cheaper bi-encoder or hybrid search already narrowed down.


### 🧪 Try it yourself

Write a query where the initial dense retrieval order and the reranked order actually *differ* —
i.e. reranking changes which document ends up #1. (Hint: try a query containing an exact phrase
from one of the *lower*-ranked documents in dense retrieval — the `substring_bonus` should help it
climb.)


In [31]:
# TODO Your turn -- find a query where reranking changes the #1 result
my_query = "more accurate but slower model"  # exact phrase from CORPUS_V1[4]

initial = [doc for _, doc in dense_retrieve(my_query, CORPUS_V1, embedder, k=5)]
print("Dense retrieval #1:", initial[0][:70], "...")

reranked = rerank(my_query, initial)
print("Reranked #1:       ", reranked[0][1][:70], "...")

print("\nDid reranking change the top result?", initial[0] != reranked[0][1])


Dense retrieval #1: A reranker takes the shortlist from initial retrieval and re-scores it ...
Reranked #1:        A reranker takes the shortlist from initial retrieval and re-scores it ...

Did reranking change the top result? False


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Whether the top result actually changes depends on how close the dense-retrieval scores were to
begin with — if TF-IDF already ranked the exact-phrase document #1 (likely, here, since "more
accurate but slower model" shares heavy vocabulary with `CORPUS_V1[4]`), reranking will *confirm*
rather than *change* the top result. To reliably see reranking flip the order, try a query that
dense retrieval ranks *ambiguously* — e.g. one where two documents have close cosine similarity
scores — since that's exactly where the reranker's extra signal (exact substring and bigram
matches) tips the balance.

This is a useful real-world lesson too: reranking earns its cost specifically on borderline cases.
When initial retrieval is already confident and correct, a reranker mostly just re-confirms the
same order — the value shows up on the harder, closer calls.
</details>


## Chapter 10 — Prompt Engineering & Context Construction for RAG

Retrieving the right chunks is only half the job — how they're assembled into a prompt matters
too. Three specific issues repeatedly show up in production RAG systems:

1. **"Lost in the Middle"** — LLMs are noticeably better at using information at the very start or
   end of a long prompt than information buried in the middle. Dumping retrieved chunks in
   similarity-rank order isn't optimal.
2. **No citation instructions** — without being told to, models blend retrieved facts with their
   own general knowledge in ways users can't distinguish.
3. **No "I don't know" guardrail** — as we saw directly in Chapter 7's exercise, a model (or a toy
   retrieval score) will often confidently answer even when nothing relevant was actually found.

Let's build a production-style prompt template that addresses all three.


In [32]:
# Chapter 10 -- A production-style grounded prompt template

GROUNDED_PROMPT_TEMPLATE = """You are a helpful assistant that answers questions using ONLY the numbered sources below.

RULES:
1. Cite the source number (like [1]) for every claim you make.
2. If the sources do not contain enough information to answer, say "I don't have enough information to answer this" -- do NOT guess or use outside knowledge.
3. Keep the answer concise and directly grounded in the cited sources.

SOURCES:
{sources}

QUESTION: {question}

ANSWER (with citations):"""

def build_grounded_prompt(query, retrieved_chunks, min_relevance_score=None):
    """
    retrieved_chunks: list of dicts with at least a 'text' key (and optionally 'score').
    min_relevance_score: if set, chunks below this score are dropped BEFORE prompt construction --
    this is the guardrail against feeding the model junk context.
    """
    if min_relevance_score is not None:
        retrieved_chunks = [c for c in retrieved_chunks if c.get("score", 1.0) >= min_relevance_score]

    if not retrieved_chunks:
        # No prompt needed at all -- short-circuit before ever calling the LLM.
        return None

    # Best-first, worst-last ordering combats "lost in the middle": if we can't fit everything,
    # what's cut is the middle-ranked material, not the most relevant piece.
    sources_text = "\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(retrieved_chunks))
    return GROUNDED_PROMPT_TEMPLATE.format(sources=sources_text, question=query)


retrieved = index.search("how does structure-aware chunking help", k=3)
prompt = build_grounded_prompt("How does structure-aware chunking help technical documents?", retrieved)
print(prompt)


You are a helpful assistant that answers questions using ONLY the numbered sources below.

RULES:
1. Cite the source number (like [1]) for every claim you make.
2. If the sources do not contain enough information to answer, say "I don't have enough information to answer this" -- do NOT guess or use outside knowledge.
3. Keep the answer concise and directly grounded in the cited sources.

SOURCES:
[1] Structure-aware chunking respects headings and sections, which helps technical documents where
a table or a numbered list should not be split across chunk boundaries.
[2] Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant part. If chunks are too small,
[3] RAG systems ground LLM outputs in retrieved evidence. This reduces hallucination compared to
closed-book generation, because the model has real, traceable source text to draw from rather
than relying purely on its parametric memory.


### The guardrail in action

Now let's see the `min_relevance_score` guardrail actually short-circuit prompt construction —
exactly the capability Chapter 7's naive pipeline was missing.


In [33]:
# Chapter 10 -- Demonstrating the guardrail: no prompt is built at all if nothing clears the bar

off_topic_results = index.search("what is the boiling point of mercury", k=3)
print("Retrieved scores:", [round(r["score"], 3) for r in off_topic_results])

# Setting a high bar to force the guardrail to trigger for this demonstration
prompt_or_none = build_grounded_prompt(
    "What is the boiling point of mercury?", off_topic_results, min_relevance_score=0.9
)

if prompt_or_none is None:
    print("\nNo prompt was built -- the pipeline can short-circuit straight to")
    print("'I don't have enough information' WITHOUT ever calling the (expensive) LLM.")
else:
    print(prompt_or_none)


Retrieved scores: [0.31, 0.185, 0.128]

No prompt was built -- the pipeline can short-circuit straight to
'I don't have enough information' WITHOUT ever calling the (expensive) LLM.


### Context token budgets

Every generator model has a maximum context length, and every retrieved chunk consumes part of
that budget. Production systems set an explicit token budget and truncate or drop lower-ranked
chunks to stay within it. Below, we simulate a simple word-count-based budget (a rough stand-in for
a real tokenizer, which would count sub-word tokens rather than whole words).

> **🏭 Production note:** Real systems use the *actual* tokenizer for their chosen model (e.g.
> `tiktoken` for OpenAI models) to count tokens precisely, since "tokens" and "words" aren't the
> same thing — this notebook's word-count approximation is only for building intuition.


In [34]:
# Chapter 10 -- A simple context token (word-count) budget enforcer

def enforce_token_budget(retrieved_chunks, max_words=40):
    """Greedily keep highest-ranked chunks until the word budget is used up."""
    kept = []
    used = 0
    for chunk in retrieved_chunks:  # assumed already sorted best-first
        n_words = len(chunk["text"].split())
        if used + n_words > max_words:
            continue  # skip this chunk, but keep checking lower-ranked (possibly shorter) ones
        kept.append(chunk)
        used += n_words
    return kept, used

wide_results = index.search("chunking and embeddings", k=6)
print(f"Retrieved {len(wide_results)} chunks, total words:",
      sum(len(r["text"].split()) for r in wide_results))

budgeted, used_words = enforce_token_budget(wide_results, max_words=40)
print(f"After a 40-word budget: kept {len(budgeted)} chunks, using {used_words} words")
for c in budgeted:
    print(f"  - ({len(c['text'].split())} words) {c['text'][:60]}...")


Retrieved 6 chunks, total words: 168
After a 40-word budget: kept 1 chunks, using 24 words
  - (24 words) Structure-aware chunking respects headings and sections, whi...


### 🧪 Try it yourself

`enforce_token_budget` above is deliberately simple: it processes chunks in order and skips any
that don't fit, but it doesn't try to backfill with a smaller lower-ranked chunk after a big one is
skipped in a way that reorders them. Trace through what happens if the *2nd*-ranked chunk is huge
(doesn't fit) but the *3rd*-ranked chunk is small (does fit) — does the 3rd chunk make it in? Test
it by constructing a small example by hand.


In [35]:
# TODO Your turn -- construct a hand-made example to test the budget-skipping behavior
test_chunks = [
    {"text": "short first chunk here"},                                          # 4 words, rank 1
    {"text": " ".join(["padding"] * 50)},                                         # 50 words, rank 2 (too big)
    {"text": "small third chunk fits easily"},                                    # 5 words, rank 3
]

kept, used = enforce_token_budget(test_chunks, max_words=15)
print(f"Kept {len(kept)} of {len(test_chunks)} chunks, using {used} words:")
for c in kept:
    print(" -", c["text"][:50])


Kept 2 of 3 chunks, using 9 words:
 - short first chunk here
 - small third chunk fits easily


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Yes — the 3rd-ranked chunk **does** make it in, even though the 2nd-ranked chunk was skipped. The
loop is a `continue`, not a `break`: when a chunk doesn't fit, it's simply skipped, and the loop
keeps checking every subsequent chunk against the *remaining* budget. So you should see both the
1st chunk (4 words) and 3rd chunk (5 words) kept, using 9 of the 15-word budget, while the bulky
2nd chunk is dropped entirely.

This "skip and keep checking" behavior is a real, deliberate design choice, not just this notebook's
simplification — it maximizes how much *relevant* content fits in a fixed budget rather than
stopping greedily at the first chunk that happens to be too big. The trade-off is that it can
produce a slightly odd-looking final order (chunk 1, then chunk 3, with chunk 2 silently missing) —
production systems sometimes add a note like "[2 lower-relevance chunks omitted for length]" so the
model (and a human reviewing the prompt) isn't confused by the gap.
</details>


## Chapter 11 — Advanced Architectures I: Self-RAG, Corrective RAG (CRAG), Adaptive RAG

Naive RAG (Chapter 7) retrieves and generates the same way every time, regardless of whether
retrieval was actually needed or whether what was retrieved is any good. Three techniques add
different forms of self-awareness:

- **Self-RAG** (Asai et al., 2023/2024) — the generator model itself produces special "reflection
  tokens" deciding whether to retrieve, whether a passage is relevant, and whether its own output
  is well-supported. Requires retraining the model.
- **Corrective RAG / CRAG** (Yan et al., 2024) — a lightweight, *separate* evaluator scores each
  retrieved document's relevance and triggers a corrective action (use as-is, discard and search
  the web, or hedge between both). No retraining needed — this is the one we'll build fully.
- **Adaptive-RAG** (Jeong et al., NAACL 2024) — routes each query to the cheapest strategy that can
  answer it, based on predicted question complexity.

We'll implement a simplified CRAG loop end to end, since it's the most practical to build without a
real LLM: a toy relevance grader plus the branch logic that decides what to do next.


In [36]:
# Chapter 11 -- Simplified Corrective RAG (CRAG) loop

def grade_relevance(query, chunk_text):
    """
    Toy grader: token overlap ratio -> a CORRECT / AMBIGUOUS / INCORRECT label.
    Real systems use a small fine-tuned classifier or an LLM-as-judge for this step.
    """
    q = set(query.lower().split())
    d = set(chunk_text.lower().split())
    overlap = len(q & d) / max(len(q), 1)
    if overlap > 0.5:
        return "CORRECT"
    elif overlap > 0.15:
        return "AMBIGUOUS"
    else:
        return "INCORRECT"

def decompose_recompose(chunk_text, query):
    """CORRECT branch: strip out sentences that don't share any words with the query."""
    sentences = re.split(r"(?<=[.!?])\s+", chunk_text)
    q_words = set(query.lower().split())
    kept = [s for s in sentences if set(s.lower().split()) & q_words]
    return " ".join(kept) if kept else chunk_text

def simulate_web_search(query):
    """Stand-in for a real web search API call in the INCORRECT branch."""
    return f"[simulated web search result for: {query!r}]"

def crag_pipeline(query, index, k=3):
    retrieved = index.search(query, k=k)
    grades = [grade_relevance(query, r["text"]) for r in retrieved]

    if "CORRECT" in grades:
        # At least one solid match -- refine it and use it.
        best = retrieved[grades.index("CORRECT")]
        context = decompose_recompose(best["text"], query)
        action = "USE_REFINED_CONTEXT"
    elif "INCORRECT" not in grades or all(g == "INCORRECT" for g in grades):
        # Nothing usable at all -- fall back to external search.
        context = simulate_web_search(query)
        action = "FALLBACK_WEB_SEARCH"
    else:
        # Mixed signal -- hedge: combine refined local context with a web search.
        best = retrieved[0]
        context = decompose_recompose(best["text"], query) + " " + simulate_web_search(query)
        action = "REFINE_AND_RECOMPOSE"

    return {"query": query, "grades": grades, "action_taken": action, "context_used": context}


test_queries = [
    "What problems does chunking cause if chunks are too large?",  # naive_index covers this well
    "What was Acme Corp's Q3 2026 revenue?",                        # totally absent from naive_index
]

for q in test_queries:
    result = crag_pipeline(q, naive_index)
    print(f"QUERY: {q}")
    print(f"  grades: {result['grades']}")
    print(f"  action: {result['action_taken']}")
    print(f"  context used: {result['context_used'][:100]}...")
    print()


QUERY: What problems does chunking cause if chunks are too large?
  grades: ['CORRECT', 'INCORRECT', 'INCORRECT']
  action: USE_REFINED_CONTEXT
  context used: Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
becau...

QUERY: What was Acme Corp's Q3 2026 revenue?
  grades: ['INCORRECT', 'INCORRECT', 'INCORRECT']
  action: FALLBACK_WEB_SEARCH
  context used: [simulated web search result for: "What was Acme Corp's Q3 2026 revenue?"]...



Notice the two branches triggered above: a well-covered query gets `USE_REFINED_CONTEXT` (the
matching chunk is trimmed down to only the sentences sharing query vocabulary), while a
completely-uncovered query correctly triggers `FALLBACK_WEB_SEARCH`. To see the third
(`REFINE_AND_RECOMPOSE`, the "hedge" branch), we need a query where the grades are mixed — some
chunks correct, some incorrect, but not unanimously either way.

> **Research Foundation:** Yan, S.-Q., Gu, J.-C., Zhu, Y., & Ling, Z.-H. (2024). *Corrective
> Retrieval Augmented Generation.* arXiv:2401.15884. The retrieval evaluator is described as
> "lightweight" specifically so it can run cheaply as an extra step on top of any existing RAG
> pipeline, without retraining the generator.


In [37]:
# Chapter 11 -- Triggering the third branch: REFINE_AND_RECOMPOSE (mixed/ambiguous grades)

mixed_query = "What is the relationship between structure and precision?"
mixed_result = crag_pipeline(mixed_query, naive_index)

print(f"QUERY: {mixed_query}")
print(f"  grades: {mixed_result['grades']}")
print(f"  action: {mixed_result['action_taken']}")
print(f"  context used: {mixed_result['context_used'][:160]}...")


QUERY: What is the relationship between structure and precision?
  grades: ['AMBIGUOUS', 'AMBIGUOUS', 'INCORRECT']
  action: REFINE_AND_RECOMPOSE
  context used: Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant par...


With no `CORRECT` grade but also not *all* `INCORRECT`, the pipeline hedges: it combines a refined
version of the best available local chunk with a simulated web search result, rather than
confidently committing to either source alone. This mirrors the real CRAG paper's design — when
the evaluator is uncertain, blending sources is safer than trusting a shaky local match completely
*or* discarding potentially-useful local context entirely.


### 🧪 Try it yourself

The `grade_relevance` thresholds (`0.5` for CORRECT, `0.15` for AMBIGUOUS) were chosen somewhat
arbitrarily. Try lowering the CORRECT threshold to `0.3` and re-running the three example queries
above. Does any query's *action* change as a result? What does this tell you about how sensitive a
CRAG-style system is to how its evaluator is calibrated?


In [38]:
# TODO Your turn -- lower the CORRECT threshold and see if any action changes

def grade_relevance_v2(query, chunk_text):
    q = set(query.lower().split())
    d = set(chunk_text.lower().split())
    overlap = len(q & d) / max(len(q), 1)
    if overlap > 0.3:      # lowered from 0.5
        return "CORRECT"
    elif overlap > 0.15:
        return "AMBIGUOUS"
    else:
        return "INCORRECT"

# Temporarily swap in the new grader
_original_grader = grade_relevance
grade_relevance = grade_relevance_v2

for q in [
    "What problems does chunking cause if chunks are too large?",
    "What was Acme Corp's Q3 2026 revenue?",
    "What is the relationship between structure and precision?",
]:
    r = crag_pipeline(q, naive_index)
    print(f"{q}\n  grades={r['grades']}  action={r['action_taken']}\n")

grade_relevance = _original_grader  # restore for later chapters


What problems does chunking cause if chunks are too large?
  grades=['CORRECT', 'INCORRECT', 'INCORRECT']  action=USE_REFINED_CONTEXT

What was Acme Corp's Q3 2026 revenue?
  grades=['INCORRECT', 'INCORRECT', 'INCORRECT']  action=FALLBACK_WEB_SEARCH

What is the relationship between structure and precision?
  grades=['AMBIGUOUS', 'CORRECT', 'INCORRECT']  action=USE_REFINED_CONTEXT



<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Lowering the CORRECT threshold from `0.5` to `0.3` changes the action for the "structure and
precision" query: what was previously `AMBIGUOUS, AMBIGUOUS, INCORRECT` (triggering the
`REFINE_AND_RECOMPOSE` hedge branch) becomes `AMBIGUOUS, CORRECT, INCORRECT` — one chunk now clears
the lower bar, so the pipeline commits fully to `USE_REFINED_CONTEXT` instead of hedging with a web
search. The other two queries' actions stay the same, since their grades weren't close to either
threshold boundary.

This is exactly the point of the exercise: **a CRAG-style system is only as good as its evaluator's
calibration.** A threshold that's too loose will over-trust marginal local matches; one that's too
strict will over-trigger expensive fallback searches unnecessarily. In the real paper, this
evaluator is a trained classifier (not a hand-tuned word-overlap ratio), specifically so its
calibration is learned from data rather than guessed — but the sensitivity you just observed is a
real property of the architecture, not an artifact of our toy simplification.
</details>


## Chapter 12 — Advanced Architectures II: GraphRAG & Knowledge-Graph RAG

Every retrieval method so far — sparse, dense, hybrid — retrieves chunks *independently* based on
similarity to a query. This breaks down for relational or multi-hop questions: *"How are these two
findings connected?"* or *"Summarize everything the corpus says about a broad theme."* No single
chunk answers questions like these — the answer lives in the *relationships between* chunks.

**GraphRAG** (Edge et al., Microsoft Research, 2024) addresses this by building an explicit
knowledge graph during indexing: extract entities and relationships, group them into communities,
and generate hierarchical summaries. We'll build a minimal version: entity/relationship extraction
from our corpus, a graph structure, and graph-based retrieval that can answer a multi-hop question
flat vector search cannot.


In [39]:
# Chapter 12 -- A tiny multi-hop corpus and a toy entity/relationship extractor
# Real GraphRAG uses an LLM for extraction; we simulate it with a hand-built rule set
# so the graph-building mechanics are fully visible without needing an API key.

GRAPH_CORPUS = [
    "Priya led the ChakshuAI shoplifting-detection model, which was built using a fine-tuned SmolVLM2.",
    "Rahul led the ChakshuAI vandalism-detection model, which was also built using a fine-tuned SmolVLM2.",
    "SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memory errors on Turing GPUs.",
    "The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.",
]

# Each tuple: (entity_a, relationship, entity_b)
# In a real system, an LLM reads each document and extracts triples like these automatically.
RELATIONSHIPS = [
    ("Priya", "led", "shoplifting-detection model"),
    ("shoplifting-detection model", "built using", "SmolVLM2"),
    ("Rahul", "led", "vandalism-detection model"),
    ("vandalism-detection model", "built using", "SmolVLM2"),
    ("SmolVLM2", "replaced", "Qwen2.5-VL-3B"),
    ("Qwen2.5-VL-3B", "caused", "out-of-memory errors"),
    ("out-of-memory errors", "occurred on", "Turing GPUs"),
    ("AIRTOMS project", "runs on", "Turing GPUs"),
]

for a, rel, b in RELATIONSHIPS:
    print(f"  ({a}) --[{rel}]--> ({b})")


  (Priya) --[led]--> (shoplifting-detection model)
  (shoplifting-detection model) --[built using]--> (SmolVLM2)
  (Rahul) --[led]--> (vandalism-detection model)
  (vandalism-detection model) --[built using]--> (SmolVLM2)
  (SmolVLM2) --[replaced]--> (Qwen2.5-VL-3B)
  (Qwen2.5-VL-3B) --[caused]--> (out-of-memory errors)
  (out-of-memory errors) --[occurred on]--> (Turing GPUs)
  (AIRTOMS project) --[runs on]--> (Turing GPUs)


In [40]:
# Chapter 12 -- Building an in-memory knowledge graph and a graph traversal retriever

class KnowledgeGraph:
    def __init__(self):
        self.edges = defaultdict(list)  # entity -> list of (relationship, other_entity)

    def add_triple(self, entity_a, relationship, entity_b):
        self.edges[entity_a].append((relationship, entity_b))
        self.edges[entity_b].append((f"<-{relationship}-", entity_a))  # reverse edge for traversal

    def neighbors(self, entity):
        return self.edges.get(entity, [])

    def multi_hop_search(self, start_entity, max_hops=2):
        """
        Breadth-first traversal collecting every relationship chain reachable within max_hops.
        We track 'visited within THIS path' (not globally) so the same entity can be reached
        again via a different chain -- e.g. both Priya's and Rahul's models lead to SmolVLM2,
        and we want to be able to continue past SmolVLM2 on each of those chains separately.
        """
        paths = []
        frontier = [(start_entity, [], {start_entity})]
        for hop in range(max_hops):
            next_frontier = []
            for entity, path_so_far, visited_on_this_path in frontier:
                for rel, neighbor in self.neighbors(entity):
                    if neighbor in visited_on_this_path:
                        continue  # avoid immediate cycles within a single chain
                    new_path = path_so_far + [(entity, rel, neighbor)]
                    new_visited = visited_on_this_path | {neighbor}
                    paths.append(new_path)
                    next_frontier.append((neighbor, new_path, new_visited))
            frontier = next_frontier
        return paths


graph = KnowledgeGraph()
for a, rel, b in RELATIONSHIPS:
    graph.add_triple(a, rel, b)

print("Entities directly connected to 'SmolVLM2':")
for rel, other in graph.neighbors("SmolVLM2"):
    print(f"  --[{rel}]--> {other}")


Entities directly connected to 'SmolVLM2':
  --[<-built using-]--> shoplifting-detection model
  --[<-built using-]--> vandalism-detection model
  --[replaced]--> Qwen2.5-VL-3B


### The multi-hop question flat vector search can't answer

Now for the real test: *"What GPU issue affects both Priya's model and the AIRTOMS project?"*
This requires connecting **three separate facts** across different documents: Priya's model uses
SmolVLM2, SmolVLM2 replaced a model that caused Turing GPU errors, and AIRTOMS runs on Turing GPUs.
No single chunk in `GRAPH_CORPUS` contains this connection — let's confirm that, then show the
graph traversal finding it.


In [41]:
# Chapter 12 -- Flat vector search vs. graph traversal on a genuine multi-hop question

multi_hop_query = "What GPU issue affects both Priya's model and the AIRTOMS project?"

# 1. Flat vector search (Chapter 5's approach) over the raw corpus
graph_embedder = TfidfEmbedder()
graph_embedder.fit(GRAPH_CORPUS)
flat_results = dense_retrieve(multi_hop_query, GRAPH_CORPUS, graph_embedder, k=2)

print("FLAT VECTOR SEARCH (best single-chunk matches):")
for score, doc in flat_results:
    print(f"  score={score:.4f}  {doc}")
print("  -> No single chunk actually answers the question.\n")

# 2. Graph traversal starting from "Priya" -- follows the chain of relationships
print("GRAPH TRAVERSAL from 'Priya' (up to 5 hops):")
paths = graph.multi_hop_search("Priya", max_hops=5)
for path in paths:
    chain = " -> ".join(f"({a})--[{rel}]-->({b})" for a, rel, b in path)
    print(f"  {chain}")

print("\nThe path that actually answers the question:")
answering_path = [p for p in paths if p[-1][2] == "Turing GPUs"][0]
print("  " + " -> ".join(f"({a})--[{rel}]-->({b})" for a, rel, b in answering_path))


FLAT VECTOR SEARCH (best single-chunk matches):
  score=0.5334  The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.
  score=0.2671  Priya led the ChakshuAI shoplifting-detection model, which was built using a fine-tuned SmolVLM2.
  -> No single chunk actually answers the question.

GRAPH TRAVERSAL from 'Priya' (up to 5 hops):
  (Priya)--[led]-->(shoplifting-detection model)
  (Priya)--[led]-->(shoplifting-detection model) -> (shoplifting-detection model)--[built using]-->(SmolVLM2)
  (Priya)--[led]-->(shoplifting-detection model) -> (shoplifting-detection model)--[built using]-->(SmolVLM2) -> (SmolVLM2)--[<-built using-]-->(vandalism-detection model)
  (Priya)--[led]-->(shoplifting-detection model) -> (shoplifting-detection model)--[built using]-->(SmolVLM2) -> (SmolVLM2)--[replaced]-->(Qwen2.5-VL-3B)
  (Priya)--[led]-->(shoplifting-detection model) -> (shoplifting-detection model)--[built using]-->(SmolVLM2) -> (SmolVLM2)--[<-built using-]

Flat vector search's top match is *close* — it correctly surfaces the AIRTOMS/Turing-GPU document
— but it never connects that fact to Priya's model, because "Priya" and "Turing GPUs" never appear
together in a single chunk. Graph traversal, by contrast, walks the actual chain of relationships —
`Priya → her model → SmolVLM2 → the model it replaced → the OOM errors it caused → Turing GPUs` —
and arrives at the correct connection in five hops, none of which required any single document to
mention all the relevant facts at once.

> **Research Foundation:** Edge, D., Trinh, H., Cheng, N., Bradley, J., Chao, A., Mody, A., Truitt,
> S., & Larson, J. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused
> Summarization.* Microsoft Research. arXiv:2404.16130.

> **🏭 Production note:** Real GraphRAG uses an LLM (not hand-written rules) to extract entities and
> relationships from every document during indexing, and adds a community-detection step that
> clusters densely-connected entities and generates hierarchical summaries — enabling *global*
> queries ("summarize everything about X") in addition to the *local* traversal we just built. This
> extraction step is also GraphRAG's main cost: many LLM calls per document at indexing time, versus
> one embedding call in Naive RAG — which is why GraphRAG earns its cost specifically for
> relationship-dense corpora with genuinely multi-hop questions, not for mostly independent documents.


### 🧪 Try it yourself

Add a new relationship triple connecting "Rahul" to something else in the graph (e.g. that Rahul's
vandalism-detection model *also* depends on the Turing GPU infrastructure). Then run
`graph.multi_hop_search("Rahul", max_hops=4)` and confirm it can now find its own path to
"Turing GPUs" — proving the graph structure generalizes beyond the one path we traced by hand.


In [42]:
# TODO Your turn -- extend the graph and confirm traversal generalizes
graph.add_triple("vandalism-detection model", "trained on", "Turing GPUs")

rahul_paths = graph.multi_hop_search("Rahul", max_hops=3)
reaches_turing = [p for p in rahul_paths if p[-1][2] == "Turing GPUs"]

print(f"Paths from Rahul reaching 'Turing GPUs': {len(reaches_turing)}")
for path in reaches_turing:
    print("  " + " -> ".join(f"({a})--[{rel}]-->({b})" for a, rel, b in path))


Paths from Rahul reaching 'Turing GPUs': 1
  (Rahul)--[led]-->(vandalism-detection model) -> (vandalism-detection model)--[trained on]-->(Turing GPUs)


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

After adding the new triple, `graph.multi_hop_search("Rahul", max_hops=3)` finds exactly one path
reaching "Turing GPUs": `Rahul --[led]--> vandalism-detection model --[trained on]--> Turing GPUs`
— a direct 2-hop path, since we gave Rahul's model a direct edge to Turing GPUs rather than routing
through the SmolVLM2/Qwen2.5-VL-3B chain like Priya's did.

This demonstrates the real value of a graph structure over hard-coded logic: `multi_hop_search`
didn't need any changes to handle the new relationship — it explores whatever edges exist in the
graph, however they're connected. In a real GraphRAG system, this is exactly what makes the
approach scale: as an LLM extracts more entities and relationships from a growing document
collection, the same traversal algorithm keeps working without needing new code for each new kind
of connection.

**A subtlety worth noticing:** finding *all* paths (rather than just the shortest one) is genuinely
more expensive than standard shortest-path BFS, since the same entity can be revisited via
different chains (this is exactly the bug we fixed above — a naive "globally visited" BFS would
have missed Rahul's separate path to Turing GPUs after visiting related entities via Priya's
chain first). Real graph databases (Neo4j and similar) provide optimized traversal algorithms
for exactly this trade-off between finding all paths and finding them efficiently at scale.
</details>


## Chapter 13 — Agentic RAG: Multi-Step, Tool-Using, Iterative Retrieval

Every architecture so far — even the self-correcting ones in Chapter 11 — follows a bounded,
predetermined sequence of steps. **Agentic RAG** instead treats retrieval as one tool among several
available to an LLM operating in a reasoning loop: the model plans what information it needs,
decides which tool to use, observes the result, and decides whether it has enough evidence to
answer or needs another round. This loop — plan, act, observe, repeat — is commonly implemented
using the **ReAct** pattern (Reasoning and Acting).

Key design elements we'll build:
- **A retry budget** — a hard cap on how many cycles the loop can run, both to bound cost/latency
  and prevent looping forever on an unanswerable question.
- **Multiple tools** — not just vector search, but also a (simulated) calculator and a (simulated)
  structured lookup, since real agentic RAG systems often need more than one kind of tool.


In [43]:
# Chapter 13 -- A minimal agentic RAG loop (ReAct-style) with a retry budget and multiple tools

# Tool 1: vector search over our knowledge graph corpus from Chapter 12
def tool_search(query):
    results = dense_retrieve(query, GRAPH_CORPUS, graph_embedder, k=1)
    return results[0][1] if results else "No relevant documents found."

# Tool 2: a toy calculator, for questions requiring arithmetic over retrieved facts
def tool_calculate(expression):
    try:
        # A real agent would use a sandboxed evaluator; this is a minimal stand-in.
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "Error: invalid characters in expression."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Tool 3: a toy structured lookup (stands in for a text-to-SQL / database query tool)
PROJECT_TABLE = {
    "chakshuai": {"clips_per_class": 200, "num_classes": 5},
    "airtoms": {"corridor": "Howrah-New Delhi", "stages": 2},
}

def tool_lookup(project_name):
    return PROJECT_TABLE.get(project_name.lower().strip(), "Project not found.")


TOOLS = {
    "search": tool_search,
    "calculate": tool_calculate,
    "lookup": tool_lookup,
}

print("Available tools:", list(TOOLS.keys()))
print("\ntool_lookup('ChakshuAI') ->", tool_lookup("ChakshuAI"))
print("tool_calculate('200 * 5') ->", tool_calculate("200 * 5"))


Available tools: ['search', 'calculate', 'lookup']

tool_lookup('ChakshuAI') -> {'clips_per_class': 200, 'num_classes': 5}
tool_calculate('200 * 5') -> 1000


### The reasoning loop

A real agentic system uses an LLM to decide, at each step, which tool to call and with what
arguments, based on the question and everything observed so far. Without a real LLM, we'll
simulate the "planning" step with a small rule-based router — the loop structure (plan → act →
observe → decide whether to continue) is identical to a real system; only the "brain" making each
decision is simplified.


In [44]:
# Chapter 13 -- The ReAct-style loop: plan -> act -> observe -> decide, with a retry budget

def toy_planner(question, observations):
    """
    Stand-in for an LLM's planning step. Decides the next tool call based on simple keyword
    rules and what's already been observed. Returns (tool_name, tool_input) or ("finish", answer).
    """
    q = question.lower()

    # If we need a number of clips PER CLASS and total classes, and haven't looked them up yet
    if "total" in q and "clips" in q and "lookup:ChakshuAI" not in observations:
        return ("lookup", "ChakshuAI")

    if "lookup:ChakshuAI" in observations and "calculate" not in observations:
        data = observations["lookup:ChakshuAI"]
        expr = f"{data['clips_per_class']} * {data['num_classes']}"
        return ("calculate", expr)

    if "calculate" in observations:
        return ("finish", f"Total clips needed = {observations['calculate']}")

    # Fallback: just search
    if "search" not in observations:
        return ("search", question)

    return ("finish", f"Based on search: {observations['search']}")


def run_agentic_loop(question, max_steps=5, verbose=True):
    observations = {}
    for step in range(1, max_steps + 1):
        tool_name, tool_input = toy_planner(question, observations)

        if tool_name == "finish":
            if verbose:
                print(f"Step {step}: FINISH -> {tool_input}")
            return {"answer": tool_input, "steps_used": step, "observations": observations}

        result = TOOLS[tool_name](tool_input)
        obs_key = f"{tool_name}:{tool_input}" if tool_name == "lookup" else tool_name
        observations[obs_key] = result

        if verbose:
            print(f"Step {step}: CALL {tool_name}({tool_input!r}) -> {result}")

    return {"answer": "[retry budget exhausted -- no final answer reached]",
            "steps_used": max_steps, "observations": observations}


result = run_agentic_loop("What is the total number of clips needed across all ChakshuAI classes?")
print("\nFINAL ANSWER:", result["answer"])
print("Steps used:", result["steps_used"])


Step 1: CALL lookup('ChakshuAI') -> {'clips_per_class': 200, 'num_classes': 5}
Step 2: CALL calculate('200 * 5') -> 1000
Step 3: FINISH -> Total clips needed = 1000

FINAL ANSWER: Total clips needed = 1000
Steps used: 3


The loop correctly chains two different tools — `lookup` then `calculate` — arriving at a final
answer in 3 steps, well within the `max_steps=5` retry budget. This is the essence of agentic RAG:
retrieval (or, here, structured lookup) is just one tool the reasoning loop can reach for, and the
loop decides for itself how many steps and which tools a given question actually needs.

### A genuinely unanswerable question

Let's now probe a question none of our tools can actually answer, and watch what our (deliberately
simple) planner does with it.


In [45]:
# Chapter 13 -- Probing what happens on a genuinely unanswerable question

unanswerable = run_agentic_loop(
    "What is the average height of ChakshuAI's interns?",  # not in any tool's data
    max_steps=3,
)
print("\nFINAL ANSWER:", unanswerable["answer"])
print("Steps used:", unanswerable["steps_used"])


Step 1: CALL search("What is the average height of ChakshuAI's interns?") -> The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.
Step 2: FINISH -> Based on search: The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.

FINAL ANSWER: Based on search: The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.
Steps used: 2


Notice what actually happened: the loop finished in just 2 steps, but the answer is **wrong** —
it confidently reports an unrelated fact about Turing GPU infrastructure as if it answered a
question about intern heights, simply because that's the highest-scoring (if barely relevant)
search result. The retry budget never even got a chance to matter, because our toy planner's
fallback logic always accepts the first search result as final, with no check on whether it's
actually relevant to the question asked.

This is a genuinely important, realistic failure mode — not a contrived one. It's the exact
"ungrounded confidence" problem from Chapters 2, 7, and 11, now showing up inside an agentic loop:
having *more* reasoning steps available doesn't automatically produce *better judgment* about when
an answer is actually good enough. A production agentic RAG system needs the same kind of relevance
check CRAG uses (Chapter 11) applied at *each* step of the loop — not just a step-count budget —
or it will happily "finish" with a plausible-sounding but ungrounded answer, exactly as we just saw.

> **🏭 Production note:** Real agentic systems use the LLM itself to judge whether an observation
> actually answers the question, as part of its reasoning trace ("this result doesn't seem to
> answer the question, let me try a different tool") — which is a much stronger check than our
> toy planner's rigid rule sequence. The retry budget remains essential regardless, as a hard
> backstop against runaway cost when even good judgment can't resolve a query.

> **Design elements recap:**
> - **Retry budget** — `max_steps` bounds worst-case cost and latency.
> - **Tiered routing** — not shown here, but a real system would route simple queries to cheap
>   single-pass RAG (Chapter 7) and reserve the full agentic loop for queries that need it
>   (echoing Adaptive-RAG's philosophy from Chapter 11).
> - **Hierarchical retrieval interfaces** — exposing multiple purpose-specific tools (as we did
>   with `search`, `calculate`, `lookup`) rather than one flat search function, so the planning
>   step can reason about which tool actually fits the current sub-question.


### 🧪 Try it yourself

Add a relevance check to `run_agentic_loop`: after a `search` result comes back, only accept it as
final if it shares at least one non-trivial word with the question (similar to the CRAG grader from
Chapter 11). If it doesn't, the loop should report "I don't have enough information" instead of
finishing with an unrelated result. Re-run the intern-height question and confirm the new check
catches it.


In [46]:
# TODO Your turn -- add a relevance check before the loop accepts a search result as final

def toy_planner_v2(question, observations):
    q = question.lower()

    if "total" in q and "clips" in q and "lookup:ChakshuAI" not in observations:
        return ("lookup", "ChakshuAI")
    if "lookup:ChakshuAI" in observations and "calculate" not in observations:
        data = observations["lookup:ChakshuAI"]
        return ("calculate", f"{data['clips_per_class']} * {data['num_classes']}")
    if "calculate" in observations:
        return ("finish", f"Total clips needed = {observations['calculate']}")

    if "search" not in observations:
        return ("search", question)

    # NEW: relevance check before trusting the search result
    search_result = observations["search"]
    q_words = set(re.findall(r"[a-z0-9]+", q))
    result_words = set(re.findall(r"[a-z0-9]+", search_result.lower()))
    # Ignore very common words so overlap reflects real topical relevance
    stopwords = {"the", "a", "an", "is", "of", "what", "how", "does", "do", "to"}
    meaningful_overlap = (q_words & result_words) - stopwords

    if not meaningful_overlap:
        return ("finish", "I don't have enough information to answer this.")
    return ("finish", f"Based on search: {search_result}")


def run_agentic_loop_v2(question, max_steps=5, verbose=True):
    observations = {}
    for step in range(1, max_steps + 1):
        tool_name, tool_input = toy_planner_v2(question, observations)
        if tool_name == "finish":
            if verbose:
                print(f"Step {step}: FINISH -> {tool_input}")
            return {"answer": tool_input, "steps_used": step}
        result = TOOLS[tool_name](tool_input)
        obs_key = f"{tool_name}:{tool_input}" if tool_name == "lookup" else tool_name
        observations[obs_key] = result
        if verbose:
            print(f"Step {step}: CALL {tool_name}({tool_input!r}) -> {result}")
    return {"answer": "[retry budget exhausted]", "steps_used": max_steps}


result_v2 = run_agentic_loop_v2("What is the average height of ChakshuAI's interns?")
print("\nFINAL ANSWER:", result_v2["answer"])


Step 1: CALL search("What is the average height of ChakshuAI's interns?") -> The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.
Step 2: FINISH -> Based on search: The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.

FINAL ANSWER: Based on search: The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Surprisingly, the new relevance check **still doesn't catch it** — the loop still confidently
returns the Turing GPU fact as if it answered a question about intern heights. Trace through why:
the question and the (wrong) search result share exactly one non-stopword: `"chakshuai"`. That's
enough to pass the `meaningful_overlap` check, even though the *topic* of the result (GPU
infrastructure) has nothing to do with the *topic* of the question (average height).

This is a genuinely important, realistic lesson, not a contrived one: **word-overlap relevance
checks are exactly as blunt as the retrieval methods built on the same idea** (recall Chapter 1's
`naive_retrieve` and Chapter 7's TF-IDF threshold). A single shared keyword — especially a
topically "sticky" one like a project name that appears in many unrelated documents — is not strong
evidence of genuine relevance. This is precisely why Chapter 11's real CRAG grader and Chapter 9's
cross-encoder reranker exist: they're built specifically to catch cases like this one, where naive
lexical overlap gives a false positive.

**A working fix** would need a stricter check — for example, requiring overlap on a *meaningful
fraction* of the question's content words (not just one), or using an actual relevance classifier
rather than any keyword-overlap heuristic. Try adjusting the check to require at least half of the
question's non-stopword terms to appear in the result, and see if that's strict enough to correctly
reject this case while still accepting genuinely relevant ones.
</details>


## Chapter 14 — Multimodal & Structured-Data RAG

Real document collections are rarely pure text: PDFs mix paragraphs with tables and images;
organizational data often lives in structured databases rather than free text. When a query needs
information from a database instead of documents, retrieval looks completely different: an LLM
translates the question into a query (commonly SQL), executes it, and uses the returned rows as
context — this is called **text-to-SQL RAG**.

A system handling both document and database sources needs a **modality router**: a classification
step, run *before* retrieval, that decides which kind of source a given question actually needs.
We already built the two underlying tools in Chapter 13 (`tool_search` for documents, `tool_lookup`
for structured data) — this chapter formalizes the routing decision between them.


In [47]:
# Chapter 14 -- A modality/tool router: classifies a query as needing structured vs. unstructured retrieval

STRUCTURED_SIGNAL_WORDS = {
    "how many", "count", "total", "average", "sum", "number of", "percentage", "rate",
}

def classify_modality(query):
    q = query.lower()
    if any(signal in q for signal in STRUCTURED_SIGNAL_WORDS):
        return "structured"
    return "unstructured"

def route_and_retrieve(query, project_table, doc_corpus, doc_embedder):
    modality = classify_modality(query)
    if modality == "structured":
        # Extract a project name crudely (real systems use an LLM for this parsing step)
        for name in project_table:
            if name in query.lower():
                return {"modality": modality, "result": project_table[name]}
        return {"modality": modality, "result": "No matching project found in structured data."}
    else:
        results = dense_retrieve(query, doc_corpus, doc_embedder, k=1)
        return {"modality": modality, "result": results[0][1] if results else "No document found."}


test_queries = [
    "How many clips per class does ChakshuAI need?",
    "What replaced Qwen2.5-VL-3B and why?",
    "What is the total number of AIRTOMS pipeline stages?",
]

for q in test_queries:
    result = route_and_retrieve(q, PROJECT_TABLE, GRAPH_CORPUS, graph_embedder)
    print(f"QUERY: {q}")
    print(f"  routed to: {result['modality']}")
    print(f"  result: {result['result']}")
    print()


QUERY: How many clips per class does ChakshuAI need?
  routed to: structured
  result: {'clips_per_class': 200, 'num_classes': 5}

QUERY: What replaced Qwen2.5-VL-3B and why?
  routed to: unstructured
  result: SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memory errors on Turing GPUs.

QUERY: What is the total number of AIRTOMS pipeline stages?
  routed to: structured
  result: {'corridor': 'Howrah-New Delhi', 'stages': 2}



All three queries route correctly: the two questions containing counting/aggregation language
("how many," "total number of") get sent to the structured lookup and return exact, correct
numeric answers rather than an approximate text passage; the question asking about a causal
relationship gets sent to document search and correctly retrieves the relevant unstructured
sentence.

> **🏭 Production note:** Real modality routers use an LLM to classify the query (much more
> robustly than our keyword list) and real text-to-SQL systems generate an actual SQL query against
> a live database schema:
> ```sql
> SELECT clips_per_class, num_classes FROM projects WHERE name = 'ChakshuAI';
> ```
> The router itself becomes a genuine engineering component in systems spanning multiple source
> types — UniversalRAG-style federated retrieval (Chapter 20) generalizes this same idea to routing
> across many separate corpora, not just two modalities.


### 🧪 Try it yourself

`classify_modality` uses a fixed list of signal phrases. Try a query that clearly *should* route to
structured data but doesn't use any of the listed signal words — e.g. "List the AIRTOMS corridor
name" (asking for a specific field value, not a count). Does it route correctly? What does this
tell you about the limits of keyword-based routing versus a real LLM-based classifier?


In [48]:
# TODO Your turn -- test a query that should route to structured data but has no signal word
tricky = "List the AIRTOMS corridor name"
result = route_and_retrieve(tricky, PROJECT_TABLE, GRAPH_CORPUS, graph_embedder)

print(f"QUERY: {tricky}")
print(f"  routed to: {result['modality']}")
print(f"  result: {result['result']}")


QUERY: List the AIRTOMS corridor name
  routed to: unstructured
  result: The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training.


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

This query **misroutes** — it goes to "unstructured" and returns an unrelated document (a sentence
about GPU infrastructure) instead of correctly pulling `{"corridor": "Howrah-New Delhi", ...}` from
`PROJECT_TABLE`. "List the AIRTOMS corridor name" is clearly a request for a specific structured
field, but it contains none of the counting/aggregation phrases our `STRUCTURED_SIGNAL_WORDS` list
was built around.

This exposes the core limitation of any fixed-keyword-list router: it can only recognize the
patterns its author anticipated in advance. Real production systems handle this with an LLM-based
classifier that understands *intent* rather than matching surface phrases — it can recognize that
"list the corridor name," "what corridor does AIRTOMS use," and "AIRTOMS corridor?" are all asking
for the same structured field, despite sharing almost no vocabulary with each other or with any
hand-written signal-word list. This is the same fundamental gap between keyword matching and
semantic understanding we first saw with BM25 vs. embeddings in Chapter 8 — it shows up again here,
one layer up, in the routing decision itself rather than in retrieval.
</details>


## Chapter 15 — RAPTOR, Late-Interaction (ColBERT) & Long-Context vs RAG

Every chunking approach so far uses one fixed granularity — great for narrow factoid questions,
poor for broad "summarize everything about X" questions. **RAPTOR** (Sarthi et al., 2024) fixes
this by building a *tree*: leaf nodes are fine-grained chunks; each level up is an LLM-generated
summary of a cluster of nodes below it, recursively, until one root summary remains. Retrieval can
then pull from whichever level matches the question's scope.

We'll build a minimal RAPTOR-style tree using our GRAPH_CORPUS, with a toy "summarizer" (real
systems use an LLM here) that just concatenates and truncates — enough to see the tree structure
and level-based retrieval clearly.


In [49]:
# Chapter 15 -- Minimal RAPTOR-style recursive summarization tree

def mock_summarize(chunks):
    # Real system: an LLM call. Toy stand-in: naive extractive concatenation, truncated.
    joined = " ".join(chunks)
    return joined[:120] + ("..." if len(joined) > 120 else "")

def cluster_pairs(nodes):
    """Toy clustering: just pair up adjacent nodes. Real RAPTOR uses embedding-based clustering
    (e.g. Gaussian Mixture Models on UMAP-reduced embeddings)."""
    pairs = []
    for i in range(0, len(nodes) - 1, 2):
        pairs.append([nodes[i], nodes[i + 1]])
    if len(nodes) % 2 == 1:
        pairs.append([nodes[-1]])
    return pairs

def build_raptor_tree(chunks, min_top_level_size=1):
    tree = {"level_0": list(chunks)}
    level = list(chunks)
    depth = 0
    while len(level) > min_top_level_size:
        depth += 1
        clusters = cluster_pairs(level)
        summaries = [mock_summarize(cluster) for cluster in clusters]
        tree[f"level_{depth}"] = summaries
        level = summaries
    return tree


raptor_tree = build_raptor_tree(GRAPH_CORPUS)

for level_name, nodes in raptor_tree.items():
    print(f"{level_name}: {len(nodes)} node(s)")
    for node in nodes:
        print(f"   - {node[:80]}")
    print()


level_0: 4 node(s)
   - Priya led the ChakshuAI shoplifting-detection model, which was built using a fin
   - Rahul led the ChakshuAI vandalism-detection model, which was also built using a 
   - SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memory errors 
   - The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuA

level_1: 2 node(s)
   - Priya led the ChakshuAI shoplifting-detection model, which was built using a fin
   - SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memory errors 

level_2: 1 node(s)
   - Priya led the ChakshuAI shoplifting-detection model, which was built using a fin



### Retrieving from the right level

The tree has 3 levels: 4 leaf chunks, 2 mid-level summaries, and 1 root summary. A narrow factoid
question should retrieve from the leaves; a broad question should retrieve from a higher level. We
don't have a real query-complexity classifier, so we'll demonstrate by embedding and searching each
level separately and comparing what comes back.


In [50]:
# Chapter 15 -- Searching different tree levels for different question scopes

def search_raptor_level(query, tree, level_name, top_k=1):
    nodes = tree[level_name]
    level_embedder = TfidfEmbedder()
    level_embedder.fit(nodes)
    return dense_retrieve(query, nodes, level_embedder, k=min(top_k, len(nodes)))

narrow_query = "What model replaced Qwen2.5-VL-3B?"
broad_query = "Give me a high-level summary of everything in this project"

print("NARROW QUESTION:", narrow_query)
print("  -> best match at level_0 (leaves):")
for score, node in search_raptor_level(narrow_query, raptor_tree, "level_0"):
    print(f"     score={score:.4f}  {node[:90]}")

print("\nBROAD QUESTION:", broad_query)
print("  -> best match at level_2 (root):")
for score, node in search_raptor_level(broad_query, raptor_tree, "level_2"):
    print(f"     score={score:.4f}  {node[:90]}")


NARROW QUESTION: What model replaced Qwen2.5-VL-3B?
  -> best match at level_0 (leaves):
     score=0.7705  SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memory errors on Turing 

BROAD QUESTION: Give me a high-level summary of everything in this project
  -> best match at level_2 (root):
     score=0.2085  Priya led the ChakshuAI shoplifting-detection model, which was built using a fine-tuned Sm


The narrow question finds an exact, highly-scored match at the leaf level, precisely because
leaves preserve the specific fact. The root-level match for the broad question is necessarily
coarser (it's a 120-character truncated concatenation, in our toy summarizer) — but in a real
system with an actual LLM-generated summary, this is exactly the level a genuinely broad question
should draw from, rather than being forced to piece together an answer from disconnected leaves.

> **Research Foundation:** Sarthi, P., Abdullah, S., Tuli, A., Khanna, S., Goldie, A., & Manning,
> C. D. (2024). *RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval.* ICLR 2024.
> arXiv:2401.18059.

### Late-interaction retrieval: ColBERT (concept only)

Standard dense retrieval compresses an entire document into **one** vector, losing fine-grained,
token-level detail. **ColBERT** (Khattab & Zaharia, 2020) keeps a separate embedding for *every
token* in both query and document, and scores relevance with **MaxSim**: for each query token, find
its single best-matching document token, then sum those best-matches across all query tokens.
Implementing real ColBERT requires a trained token-level embedding model, which is out of scope for
this dependency-free notebook — but the MaxSim scoring *logic* itself is simple enough to illustrate
directly with our existing TF-IDF word vectors, treating each word as its own "token embedding."


In [51]:
# Chapter 15 -- Illustrating MaxSim scoring logic (the core idea behind ColBERT's late interaction)
# NOTE: this is a simplified illustration of the SCORING MECHANISM, not a real ColBERT
# implementation -- real ColBERT uses a trained transformer to produce per-token embeddings.

def token_overlap_similarity(token_a, token_b):
    # Toy per-token "embedding similarity": exact match = 1.0, no match = 0.0.
    # A real system would use actual token embeddings and cosine similarity here.
    return 1.0 if token_a == token_b else 0.0

def maxsim_score(query, document):
    query_tokens = re.findall(r"[a-z0-9]+", query.lower())
    doc_tokens = re.findall(r"[a-z0-9]+", document.lower())
    total = 0.0
    for q_tok in query_tokens:
        best = max((token_overlap_similarity(q_tok, d_tok) for d_tok in doc_tokens), default=0.0)
        total += best
    return total

query = "SmolVLM2 replaced errors"
for doc in GRAPH_CORPUS:
    score = maxsim_score(query, doc)
    print(f"MaxSim={score:.1f}  {doc[:70]}...")


MaxSim=1.0  Priya led the ChakshuAI shoplifting-detection model, which was built u...
MaxSim=1.0  Rahul led the ChakshuAI vandalism-detection model, which was also buil...
MaxSim=3.0  SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memo...
MaxSim=0.0  The AIRTOMS project runs on the same Turing GPU infrastructure used fo...


The SmolVLM2 document scores highest (3.0 — a perfect per-token match on all three query words),
clearly separated from the two "model" documents (1.0, matching only on the shared word "SmolVLM2")
and the AIRTOMS document (0.0, no overlap at all). This is the essence of MaxSim: rather than
compressing the whole document into one vector and losing which specific words drove the match,
scoring happens token by token, so a document with several strong individual-word matches is
rewarded more than one with a single shared word buried among a lot of unrelated text.

> **⚠️ A caveat about this simplification:** Our `token_overlap_similarity` only recognizes *exact*
> string matches (1.0 or 0.0) — real ColBERT uses trained token embeddings, so it recognizes
> near-synonyms and related terms at the token level too, not just identical spellings. Try the
> original query `"SmolVLM2 replaced Qwen model"` from a moment ago and notice it produces a
> **tie** between two documents — because "Qwen" doesn't exact-match "Qwen2.5-VL-3B" as tokenized
> here, and "model" happens to appear in an unrelated document. This is exactly the gap a real,
> trained token embedding model closes.

> **Research Foundation:** Khattab, O., & Zaharia, M. (2020). *ColBERT: Efficient and Effective
> Passage Search via Contextualized Late Interaction over BERT.* SIGIR 2020. arXiv:2004.12832.

### Long-context LLMs vs. RAG

With context windows now reaching hundreds of thousands of tokens, why not just skip retrieval
entirely and put the whole corpus in the prompt? In practice, RAG remains relevant even for
long-context models because of **cost** (processing a huge context on every query is expensive),
**latency** (more input tokens = slower generation), the **"lost in the middle" effect** (Chapter
10 — having information present doesn't guarantee the model uses it well), and **scale** (many real
corpora are simply too large for any context window). The practical answer: long context and RAG
are complementary, not competing — RAG narrows a large corpus down to a relevant subset, and a
longer context window gives that subset more room to include full documents rather than
aggressively truncated chunks.


### 🧪 Try it yourself

`build_raptor_tree`'s `cluster_pairs` function pairs up *adjacent* nodes without looking at their
content at all — it's a placeholder for real embedding-based clustering. Confirm this limitation
directly: shuffle `GRAPH_CORPUS` into a different order before building the tree, and check whether
the resulting `level_1` summaries group genuinely related documents together, or just whatever
happened to land next to each other after shuffling.


In [52]:
# TODO Your turn -- shuffle the corpus and see how it affects (naive) clustering
shuffled_corpus = GRAPH_CORPUS.copy()
random.shuffle(shuffled_corpus)

print("Shuffled order:")
for doc in shuffled_corpus:
    print(" -", doc[:70], "...")

shuffled_tree = build_raptor_tree(shuffled_corpus)
print("\nlevel_1 groupings from the SHUFFLED corpus:")
for node in shuffled_tree["level_1"]:
    print(" -", node[:100])


Shuffled order:
 - The AIRTOMS project runs on the same Turing GPU infrastructure used fo ...
 - Rahul led the ChakshuAI vandalism-detection model, which was also buil ...
 - Priya led the ChakshuAI shoplifting-detection model, which was built u ...
 - SmolVLM2 replaced Qwen2.5-VL-3B after Qwen2.5-VL-3B caused out-of-memo ...

level_1 groupings from the SHUFFLED corpus:
 - The AIRTOMS project runs on the same Turing GPU infrastructure used for ChakshuAI model training. Ra
 - Priya led the ChakshuAI shoplifting-detection model, which was built using a fine-tuned SmolVLM2. Sm


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

With this notebook's fixed random seed, shuffling puts the AIRTOMS document (about railway/GPU
infrastructure) next to Rahul's vandalism-detection document — two topically unrelated passages —
simply because `random.shuffle` happened to place them adjacently. `cluster_pairs` then dutifully
merges them into one `level_1` summary node, blending two unrelated topics together.

This is exactly the limitation flagged in the intro: our toy `cluster_pairs` has **no awareness of
content** — it's purely positional. Real RAPTOR clusters using the documents' *embeddings*
(typically via a Gaussian Mixture Model on UMAP-reduced vectors), so documents end up grouped by
actual semantic similarity, regardless of what order they happened to appear in the source corpus.
A production implementation would replace `cluster_pairs` with something like:
```python
from sklearn.mixture import GaussianMixture
embeddings = embedder.transform(nodes)
clusters = GaussianMixture(n_components=k).fit_predict(embeddings)
```
so that summarization groups genuinely related content, producing coherent mid-level summaries
regardless of input order — which matters a great deal, since real document collections have no
natural "adjacency" the way our small ordered list does.
</details>


## Chapter 16 — Evaluating RAG: RAGAS, ARES, DeepEval, TruLens & Custom Metrics

A RAG system can fail in two structurally different ways: **retrieval** can fail (wrong or no
relevant chunks found), or **generation** can fail (relevant chunks were retrieved, but the model
didn't use them well). Good evaluation measures both surfaces separately, because the fixes are
completely different.

We'll implement the four core RAGAS-style metrics from scratch:

| Metric | Failure surface | What it measures |
|---|---|---|
| **Context Precision** | Retrieval | Of the chunks retrieved, what fraction were actually relevant? |
| **Context Recall** | Retrieval | Of the truly relevant chunks available, what fraction did retrieval find? |
| **Faithfulness** | Generation | Is every claim in the answer supported by the retrieved context? |
| **Answer Relevancy** | Generation | Does the answer actually address the question asked? |


In [53]:
# Chapter 16 -- Simplified, from-scratch RAG evaluation metrics
# Real systems use an LLM-as-judge for these; we simulate the judgments with word-overlap
# heuristics so the metric MECHANICS are fully visible without needing an API key.

def context_precision(retrieved_texts, relevant_texts):
    """Of what was retrieved, what fraction is actually in the known-relevant set?"""
    if not retrieved_texts:
        return 0.0
    hits = sum(1 for t in retrieved_texts if t in relevant_texts)
    return hits / len(retrieved_texts)

def context_recall(retrieved_texts, relevant_texts):
    """Of what's actually relevant, what fraction did we retrieve?"""
    if not relevant_texts:
        return 1.0
    hits = sum(1 for t in relevant_texts if t in retrieved_texts)
    return hits / len(relevant_texts)

def faithfulness(answer, retrieved_texts):
    """Toy check: what fraction of the answer's content words appear somewhere in the
    retrieved context? A real system decomposes the answer into individual claims and
    checks each one against the context with an LLM judge."""
    answer_words = set(re.findall(r"[a-z0-9]+", answer.lower()))
    context_words = set()
    for t in retrieved_texts:
        context_words.update(re.findall(r"[a-z0-9]+", t.lower()))
    stopwords = {"the", "a", "an", "is", "of", "to", "and", "based", "on", "this"}
    answer_content_words = answer_words - stopwords
    if not answer_content_words:
        return 1.0
    supported = answer_content_words & context_words
    return len(supported) / len(answer_content_words)

def answer_relevancy(answer, question):
    """Toy check: how much does the answer's vocabulary overlap with the question's?
    A real system generates several hypothetical questions the answer WOULD answer and
    compares those to the actual question via embedding similarity."""
    q_words = set(re.findall(r"[a-z0-9]+", question.lower()))
    a_words = set(re.findall(r"[a-z0-9]+", answer.lower()))
    if not q_words:
        return 0.0
    return len(q_words & a_words) / len(q_words)


# Evaluate one full example end to end
question = "What problems does chunking cause if chunks are too large?"
retrieved = naive_rag_answer(question, naive_index, k=2)
retrieved_texts = [r["text"] for r in retrieved["retrieved"]]
generated_answer = retrieved["answer"]

# For this toy example, we derive "truly relevant" chunks by checking which chunks contain the
# key phrase the question is actually asking about -- a stand-in for real human-labeled ground
# truth, which is how RAGAS-style evaluation is normally seeded in practice.
relevant_texts = [c["text"] for c in structured_chunks if "too large" in c["text"]]

print("QUESTION: ", question)
print("ANSWER:   ", generated_answer[:100], "...")
print()
print(f"Context Precision : {context_precision(retrieved_texts, relevant_texts):.2f}")
print(f"Context Recall    : {context_recall(retrieved_texts, relevant_texts):.2f}")
print(f"Faithfulness      : {faithfulness(generated_answer, retrieved_texts):.2f}")
print(f"Answer Relevancy  : {answer_relevancy(generated_answer, question):.2f}")


QUESTION:  What problems does chunking cause if chunks are too large?
ANSWER:    Based on the retrieved context: "Chunking determines what unit of text gets retrieved. If chunks are ...

Context Precision : 0.50
Context Recall    : 1.00
Faithfulness      : 0.96
Answer Relevancy  : 0.70


Reading these results: Context Precision of 0.50 means only one of the two retrieved chunks was
actually relevant (the naive pipeline's `k=2` pulled in one useful chunk plus one that wasn't).
Context Recall of 1.00 means the one truly-relevant chunk in our small ground-truth set *was*
successfully retrieved. Faithfulness of 0.96 means the answer's content words are almost entirely
traceable back to the retrieved context (expected, since our toy generator just quotes retrieved
text directly). Answer Relevancy of 0.70 reflects that the answer, while grounded, doesn't restate
every word of the question.

Notice the two failure surfaces these numbers separate cleanly: a **low Context Precision** with a
**high Faithfulness** would tell you retrieval is pulling in noise, but generation is still being
honest about what it was given — the fix is better retrieval (Chapter 8 hybrid search, Chapter 9
reranking), not a different generator.

> **Research Foundation:** Es, S., James, J., Espinosa-Anke, L., & Schockaert, S. (2024). *RAGAS:
> Automated Evaluation of Retrieval Augmented Generation.* Proceedings of EACL 2024 (System
> Demonstrations), pp. 150–158. arXiv:2309.15217.
>
> Saad-Falcon, J., Khattab, O., Potts, C., & Zaharia, M. (2024). *ARES: An Automated Evaluation
> Framework for Retrieval-Augmented Generation Systems.* Proceedings of NAACL 2024, pp. 338–354.

> **🏭 Production note:** Real RAGAS-style evaluation uses an LLM as the judge for each of these
> checks (deciding faithfulness by reasoning about entailment, not counting shared words), and the
> real library is a drop-in replacement for everything above:
> ```python
> from ragas import evaluate
> from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
> result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
> ```
> DeepEval and TruLens provide similar LLM-judge-based metrics plus broader test-suite tooling and
> production monitoring — useful for tracking these same four numbers over time as your corpus,
> embedding model, or generator model changes, since a system that scored well once can silently
> degrade later.


### 🧪 Try it yourself

Run the same evaluation on the CRAG pipeline's answer from Chapter 11 (the `"What was Acme Corp's
Q3 2026 revenue?"` example, which triggered `FALLBACK_WEB_SEARCH`) instead of the naive pipeline's
answer. What does `faithfulness` report when the "context" is a simulated web search string rather
than a real retrieved chunk? Does this reveal a limitation of our toy faithfulness metric?


In [54]:
# TODO Your turn -- evaluate CRAG's fallback-search output with our faithfulness metric
crag_result = crag_pipeline("What was Acme Corp's Q3 2026 revenue?", naive_index)
print("CRAG action:", crag_result["action_taken"])
print("CRAG context_used:", crag_result["context_used"])

# CRAG's pipeline (Chapter 11) never actually built a separate "generated answer" -- it only
# produced context_used. To evaluate faithfulness we need an actual answer, so let's treat
# context_used AS the answer, exactly the way a naive generator might just echo it back.
faithfulness_score = faithfulness(crag_result["context_used"], [crag_result["context_used"]])
print("\nFaithfulness (answer checked against itself):", faithfulness_score)


CRAG action: FALLBACK_WEB_SEARCH
CRAG context_used: [simulated web search result for: "What was Acme Corp's Q3 2026 revenue?"]

Faithfulness (answer checked against itself): 1.0


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Faithfulness comes back as a perfect `1.0` — but notice this is **trivially true and not actually
informative**: we checked the string `crag_result["context_used"]` for faithfulness against
`[crag_result["context_used"]]`, i.e. the exact same string against itself. Of course every word
overlaps with itself.

This exposes a real limitation worth internalizing: **`faithfulness` only measures whether an
answer's words appear in the retrieved context — it says nothing about whether the context itself
is actually true or useful.** Here, the "context" is a placeholder string
`'[simulated web search result for: "..."]'`, not a real answer to the revenue question at all. A
real faithfulness metric would correctly report this text as "not really an answer to check
faithfulness of" — but our word-overlap-based toy metric has no way to notice that the content is
essentially empty scaffolding rather than a genuine grounded claim.

This is the same core lesson from Chapter 7's and Chapter 13's exercises, showing up a third time:
**a purely mechanical check (word overlap, in this metric's case) can be gamed by degenerate
inputs that a real evaluator (an LLM judge, or a human) would immediately recognize as
meaningless.** It's exactly why RAGAS and ARES use an LLM as the judge for faithfulness in
practice, rather than any keyword-overlap heuristic — the LLM can recognize that "this text isn't a
real answer" in a way a mechanical word-matching check fundamentally cannot.
</details>


## Chapter 17 — The Complete Taxonomy of RAG Types

Having built (a simplified version of) nearly every technique in this notebook, we can now step
back and build a single decision reference. When designing a new RAG system, the practical approach
is almost never to reach for the most advanced technique available — it's to start with Naive RAG
(Chapter 7), evaluate it (Chapter 16), identify which specific failure surface is actually
underperforming, and adopt only the technique that targets that failure.

Let's encode that decision process as a small heuristic tool: answer a few questions about your
corpus and query patterns, and get a suggested starting architecture.


In [55]:
# Chapter 17 -- The full taxonomy as a structured, queryable table

RAG_TAXONOMY = [
    {"name": "Naive RAG", "core_idea": "Chunk -> embed -> retrieve top-k -> generate",
     "best_for": "Simple single-fact QA over a clean, small corpus",
     "limitation": "No quality control on retrieved chunks; misses on vocabulary mismatch"},
    {"name": "Hybrid Search", "core_idea": "Combine sparse (BM25) and dense retrieval via RRF",
     "best_for": "Queries mixing exact terms (codes, names) with semantic meaning",
     "limitation": "Adds a second retrieval pass to tune and maintain"},
    {"name": "Reranked", "core_idea": "Cross-encoder re-scores a retrieved shortlist",
     "best_for": "Borderline/ambiguous retrieval calls where ranking precision matters",
     "limitation": "Added latency; needs a reranker model"},
    {"name": "HyDE", "core_idea": "Embed a generated hypothetical answer, not the raw query",
     "best_for": "Queries phrased very differently from how answers are written",
     "limitation": "Extra LLM call before retrieval even starts"},
    {"name": "Self-RAG", "core_idea": "Model generates reflection tokens to self-critique",
     "best_for": "Applications needing built-in factuality self-checks",
     "limitation": "Requires retraining/fine-tuning the generator model"},
    {"name": "CRAG", "core_idea": "Lightweight evaluator scores docs, triggers correction",
     "best_for": "Adding a factuality safety net on top of an existing pipeline",
     "limitation": "Evaluator quality caps overall system quality"},
    {"name": "Adaptive-RAG", "core_idea": "Routes queries by predicted complexity",
     "best_for": "Mixed workloads with both simple and multi-hop questions",
     "limitation": "Misclassified complexity routes to the wrong strategy"},
    {"name": "Agentic RAG", "core_idea": "LLM plans and iteratively uses retrieval as one tool",
     "best_for": "Multi-step research tasks needing several data sources",
     "limitation": "Highest latency and cost of any architecture here"},
    {"name": "GraphRAG", "core_idea": "Entity/relationship graph plus hierarchical summaries",
     "best_for": "Relationship-dense corpora with genuinely multi-hop questions",
     "limitation": "Expensive indexing; needs a relationship-dense corpus to pay off"},
    {"name": "RAPTOR", "core_idea": "Recursive tree of chunk summaries at multiple levels",
     "best_for": "Corpora needing both narrow factoid AND broad summary answers",
     "limitation": "Indexing cost scales with tree depth"},
    {"name": "Late-Interaction (ColBERT)", "core_idea": "Token-level embeddings scored via MaxSim",
     "best_for": "High-precision retrieval where exact phrasing matters a lot",
     "limitation": "Much higher storage than single-vector embeddings"},
    {"name": "Multimodal RAG", "core_idea": "Retrieval over images, tables, mixed-layout PDFs",
     "best_for": "Corpora with significant non-text content",
     "limitation": "Needs multimodal embeddings or lossy text conversion"},
    {"name": "Structured-Data-Aware", "core_idea": "Text-to-SQL or schema-aware structured retrieval",
     "best_for": "Questions answerable from a database rather than documents",
     "limitation": "Requires reliable schema understanding by the model"},
    {"name": "Long-Context (alternative)", "core_idea": "Skip retrieval; put the whole corpus in the prompt",
     "best_for": "Small, bounded document sets that fit comfortably in context",
     "limitation": "Cost, latency, and lost-in-the-middle at scale"},
]

print(f"{len(RAG_TAXONOMY)} architectures catalogued.\n")
for entry in RAG_TAXONOMY[:3]:
    print(f"{entry['name']}:")
    print(f"  Core idea:  {entry['core_idea']}")
    print(f"  Best for:   {entry['best_for']}")
    print(f"  Limitation: {entry['limitation']}\n")


14 architectures catalogued.

Naive RAG:
  Core idea:  Chunk -> embed -> retrieve top-k -> generate
  Best for:   Simple single-fact QA over a clean, small corpus
  Limitation: No quality control on retrieved chunks; misses on vocabulary mismatch

Hybrid Search:
  Core idea:  Combine sparse (BM25) and dense retrieval via RRF
  Best for:   Queries mixing exact terms (codes, names) with semantic meaning
  Limitation: Adds a second retrieval pass to tune and maintain

Reranked:
  Core idea:  Cross-encoder re-scores a retrieved shortlist
  Best for:   Borderline/ambiguous retrieval calls where ranking precision matters
  Limitation: Added latency; needs a reranker model



### A "which RAG type do I need?" decision helper

Rather than memorizing the table, let's encode the decision logic itself: given a few
characteristics of your use case, suggest a reasonable starting architecture.


In [56]:
# Chapter 17 -- A quick "which RAG type do I need?" decision helper (toy heuristic)

def suggest_rag_architecture(
    has_multihop_questions=False,
    has_structured_data=False,
    has_images_or_tables=False,
    needs_broad_summaries=False,
    corpus_fits_in_context=False,
    exact_terms_matter=False,
    latency_budget_is_tight=True,
):
    """A simplified decision tree. Real system selection also weighs team expertise,
    existing infrastructure, and budget -- this only reasons about the technical shape
    of the problem."""
    suggestions = []

    if corpus_fits_in_context and latency_budget_is_tight is False:
        suggestions.append("Long-Context (skip RAG entirely, if truly small & bounded)")
        return suggestions

    if has_structured_data:
        suggestions.append("Structured-Data-Aware (text-to-SQL) -- for the structured portion")

    if has_images_or_tables:
        suggestions.append("Multimodal RAG -- for the non-text portion")

    if has_multihop_questions:
        suggestions.append("GraphRAG (if relationship-dense) or Agentic RAG (if tool-diverse)")

    if needs_broad_summaries:
        suggestions.append("RAPTOR -- for multi-granularity retrieval")

    if exact_terms_matter:
        suggestions.append("Hybrid Search (BM25 + dense) -- baseline retrieval upgrade")

    if not suggestions:
        suggestions.append("Naive RAG -- start simple, evaluate (Ch.16), then upgrade only what's failing")

    return suggestions


# Try it on a scenario resembling our own GRAPH_CORPUS example from Chapter 12
recommendations = suggest_rag_architecture(
    has_multihop_questions=True,
    exact_terms_matter=True,
)
print("Recommended starting points for a multi-hop, exact-term-sensitive corpus:")
for r in recommendations:
    print(" -", r)


Recommended starting points for a multi-hop, exact-term-sensitive corpus:
 - GraphRAG (if relationship-dense) or Agentic RAG (if tool-diverse)
 - Hybrid Search (BM25 + dense) -- baseline retrieval upgrade


> **🏭 Production note:** This heuristic is deliberately simple — a real architecture decision also
> weighs team expertise, existing infrastructure, indexing budget, and query volume. Its real value
> isn't the specific recommendations; it's the discipline of asking these questions explicitly
> *before* reaching for the most sophisticated technique available. Chapter 18's pre-production
> checklist and Chapter 19's cost-engineering patterns build directly on this same discipline.


### 🧪 Try it yourself

Call `suggest_rag_architecture` with every argument at its default (i.e. just
`suggest_rag_architecture()`). What does it recommend, and does that match what Chapter 7 argued
should always be the *starting point* regardless of eventual complexity?


In [57]:
# TODO Your turn -- call with all defaults and see what comes back
default_recommendation = suggest_rag_architecture()
print(default_recommendation)


["Naive RAG -- start simple, evaluate (Ch.16), then upgrade only what's failing"]


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

With every argument at its default (no special corpus characteristics flagged), the function falls
through every `if` branch and returns exactly one recommendation: `"Naive RAG -- start simple,
evaluate (Ch.16), then upgrade only what's failing"`.

This matches Chapter 7's argument directly: Naive RAG isn't a beginner's shortcut to be embarrassed
about — it's the correct, deliberate default absent any specific evidence that a more sophisticated
technique is actually needed. The function's structure encodes this on purpose: every advanced
technique in the taxonomy requires *opting in* via some specific flag describing a real
characteristic of your problem (multi-hop questions, structured data, tight latency budgets, etc.),
and only in the total absence of any such signal does it recommend the simplest possible starting
point. This mirrors good engineering practice generally: added complexity should be justified by a
specific, identified need — not adopted by default "just in case."
</details>


## Chapter 18 — Limitations & Failure Modes of RAG Systems

RAG failure modes are easier to diagnose when organized by which **layer** of the system they
belong to, since each layer has different owners and different fixes:

- **Data layer** — stale/contradictory sources, embedding drift
- **Retrieval layer** — misses, semantic-vs-exact mismatch, multi-hop blindness, context amnesia
- **Generation layer** — prompt overload, faithfulness gaps
- **Systems/operational layer** — latency at scale, complex stack, access control, governance

We've directly *demonstrated* several of these failure modes already, in earlier chapters'
exercises. Let's build a structured, adaptable checklist that ties each item back to where it was
shown, so it's a genuine working reference rather than an abstract list.


In [58]:
# Chapter 18 -- A pre-production RAG failure-mode checklist, cross-referenced to what we've SEEN

FAILURE_MODE_CHECKLIST = [
    {"layer": "Data", "failure_mode": "Stale or contradictory sources",
     "seen_in": "Not directly demonstrated -- requires a corpus with multiple document versions",
     "mitigation": "Version documents; prefer most-recent source in retrieval ranking or metadata filter"},
    {"layer": "Data", "failure_mode": "Embedding drift after model upgrade",
     "seen_in": "Not directly demonstrated -- requires swapping embedding models mid-corpus",
     "mitigation": "Re-embed the entire corpus whenever the embedding model changes; never mix vector spaces"},
    {"layer": "Retrieval", "failure_mode": "Retrieval misses (vocabulary mismatch)",
     "seen_in": "Ch.1 exercise -- 'old-fashioned keyword search' failed to match the BM25 passage",
     "mitigation": "Use dense/hybrid retrieval (Ch.5, Ch.8) instead of pure keyword overlap"},
    {"layer": "Retrieval", "failure_mode": "Semantic-vs-exact mismatch",
     "seen_in": "Ch.8 side-by-side comparison of BM25 vs. dense rankings",
     "mitigation": "Hybrid search with RRF fusion (Ch.8)"},
    {"layer": "Retrieval", "failure_mode": "Multi-hop blindness",
     "seen_in": "Ch.12 -- flat vector search could not connect Priya's model to the Turing GPU issue",
     "mitigation": "GraphRAG (Ch.12) or agentic multi-step retrieval (Ch.13)"},
    {"layer": "Retrieval", "failure_mode": "Weak relevance checks give false positives",
     "seen_in": "Ch.13 exercise -- a shared keyword ('chakshuai') passed a relevance check despite an unrelated answer",
     "mitigation": "Cross-encoder reranking (Ch.9) or a trained CRAG-style evaluator (Ch.11), not raw word overlap"},
    {"layer": "Generation", "failure_mode": "Ungrounded confidence on off-topic queries",
     "seen_in": "Ch.7 exercise -- a mercury boiling-point question still scored above threshold on an unrelated corpus",
     "mitigation": "Explicit 'I don't know' guardrails in the prompt (Ch.10); CRAG evaluator (Ch.11)"},
    {"layer": "Generation", "failure_mode": "Prompt overload / lost in the middle",
     "seen_in": "Ch.10 -- token budget enforcement demo",
     "mitigation": "Enforce a context token budget; order chunks best-first (Ch.10)"},
    {"layer": "Generation", "failure_mode": "Evaluation metrics gamed by degenerate input",
     "seen_in": "Ch.16 exercise -- a placeholder string scored perfect 'faithfulness' against itself",
     "mitigation": "Use LLM-as-judge evaluation in production, not pure word-overlap metrics"},
    {"layer": "Systems", "failure_mode": "Naive clustering ignores content",
     "seen_in": "Ch.15 exercise -- shuffling the corpus produced a nonsensical RAPTOR grouping",
     "mitigation": "Use embedding-based clustering (e.g. GMM), never positional/order-based grouping"},
]

for item in FAILURE_MODE_CHECKLIST:
    print(f"[{item['layer']}] {item['failure_mode']}")
    print(f"   Seen in:    {item['seen_in']}")
    print(f"   Mitigation: {item['mitigation']}\n")


[Data] Stale or contradictory sources
   Seen in:    Not directly demonstrated -- requires a corpus with multiple document versions
   Mitigation: Version documents; prefer most-recent source in retrieval ranking or metadata filter

[Data] Embedding drift after model upgrade
   Seen in:    Not directly demonstrated -- requires swapping embedding models mid-corpus
   Mitigation: Re-embed the entire corpus whenever the embedding model changes; never mix vector spaces

[Retrieval] Retrieval misses (vocabulary mismatch)
   Seen in:    Ch.1 exercise -- 'old-fashioned keyword search' failed to match the BM25 passage
   Mitigation: Use dense/hybrid retrieval (Ch.5, Ch.8) instead of pure keyword overlap

[Retrieval] Semantic-vs-exact mismatch
   Seen in:    Ch.8 side-by-side comparison of BM25 vs. dense rankings
   Mitigation: Hybrid search with RRF fusion (Ch.8)

[Retrieval] Multi-hop blindness
   Seen in:    Ch.12 -- flat vector search could not connect Priya's model to the Turing GPU issue


### A simple readiness scorer

Beyond just listing failure modes, let's build a tiny self-assessment tool: for each layer, ask
whether your system has *addressed* the relevant failure modes, and get a rough readiness score.


In [59]:
# Chapter 18 -- A pre-production readiness self-assessment (adapt the answers to YOUR system)

def assess_readiness(answers):
    """
    answers: dict mapping each FAILURE_MODE_CHECKLIST item's failure_mode -> True/False
    (True = "we've mitigated this"). Returns a readiness percentage and the unmitigated list.
    """
    total = len(FAILURE_MODE_CHECKLIST)
    mitigated = sum(1 for item in FAILURE_MODE_CHECKLIST if answers.get(item["failure_mode"], False))
    unmitigated = [item["failure_mode"] for item in FAILURE_MODE_CHECKLIST
                   if not answers.get(item["failure_mode"], False)]
    return {"score_pct": round(100 * mitigated / total, 1), "unmitigated": unmitigated}


# Example: a hypothetical system that has hybrid search and prompt guardrails, but nothing else
my_system_answers = {
    "Semantic-vs-exact mismatch": True,      # we built hybrid search (Ch.8)
    "Prompt overload / lost in the middle": True,  # we enforce a token budget (Ch.10)
    "Ungrounded confidence on off-topic queries": True,  # we added an "I don't know" guardrail (Ch.10)
}

assessment = assess_readiness(my_system_answers)
print(f"Readiness score: {assessment['score_pct']}%")
print("\nStill unmitigated:")
for gap in assessment["unmitigated"]:
    print(" -", gap)


Readiness score: 30.0%

Still unmitigated:
 - Stale or contradictory sources
 - Embedding drift after model upgrade
 - Retrieval misses (vocabulary mismatch)
 - Multi-hop blindness
 - Weak relevance checks give false positives
 - Evaluation metrics gamed by degenerate input
 - Naive clustering ignores content


A 30% readiness score for a system with only three of ten mitigations in place is a useful, honest
signal — and notice that even "Retrieval misses (vocabulary mismatch)" shows up as unmitigated
despite our hypothetical system having hybrid search, because we only marked "Semantic-vs-exact
mismatch" as addressed, not the separate vocabulary-mismatch item. This is intentional: the
checklist treats related-but-distinct failure modes as genuinely separate line items, because
fixing one doesn't automatically fix the others, even when they sound similar.

> **🏭 Production note:** A checklist like this is only as good as how honestly it's filled in.
> The real value isn't the percentage — it's the explicit, itemized list of *specific* gaps,
> reviewed by the team before any production deployment, ideally with each "True" backed by an
> actual test (like the exercises throughout this notebook) rather than just an assumption that a
> given component "probably handles that."


### 🧪 Try it yourself

Add mitigations for "Multi-hop blindness" (we built GraphRAG in Chapter 12) and "Weak relevance
checks give false positives" (arguably still NOT fully solved — recall Chapter 13's exercise, where
our attempted fix didn't actually work). Should you mark that second one `True` or `False`, given
what we actually observed? Recompute the readiness score.


In [60]:
# TODO Your turn -- update the answers and recompute readiness
my_system_answers_v2 = dict(my_system_answers)  # start from the previous answers
my_system_answers_v2["Multi-hop blindness"] = True  # we built GraphRAG in Ch.12

# Careful: did our Ch.13 exercise's relevance-check fix actually work?
my_system_answers_v2["Weak relevance checks give false positives"] = False  # <-- your call here

new_assessment = assess_readiness(my_system_answers_v2)
print(f"Updated readiness score: {new_assessment['score_pct']}%")
print("\nStill unmitigated:")
for gap in new_assessment["unmitigated"]:
    print(" -", gap)


Updated readiness score: 40.0%

Still unmitigated:
 - Stale or contradictory sources
 - Embedding drift after model upgrade
 - Retrieval misses (vocabulary mismatch)
 - Weak relevance checks give false positives
 - Evaluation metrics gamed by degenerate input
 - Naive clustering ignores content


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

The updated score comes out to **40%** (4 of 10 mitigated), and "Multi-hop blindness" correctly
drops off the unmitigated list once marked `True`.

The more important part of this exercise is the judgment call on "Weak relevance checks give false
positives." Chapter 13's exercise added a relevance check specifically to catch this failure mode —
but when we actually tested it, it **did not catch** the case it was designed for (the "chakshuai"
shared-keyword false positive). Marking this `True` would be dishonest — we have a check *in place*,
but we directly observed it doesn't actually work for a realistic case. This is exactly the
distinction the production note stresses: a readiness checklist item should reflect "we tested this
and it works," not "we wrote something that looks like it should address this."

This is arguably the single most important habit this entire notebook has tried to build: **run the
code and look at what it actually outputs before trusting your assumption about what it does.**
Several exercises across this notebook (Ch.7, Ch.9, Ch.11, Ch.13, Ch.15, Ch.16) turned out to behave
differently than a first guess would suggest — and catching that gap between assumption and
observed behavior is the core skill a pre-production checklist like this one is meant to enforce
at the system level, not just the exercise level.
</details>


## Chapter 19 — Production RAG: Architecture, Scaling, Security, Governance, Cost

A mature production RAG system chains together every stage from this notebook: parsing → chunking
(Ch.4) → embeddings (Ch.5) → vector store (Ch.6) → hybrid retrieval (Ch.8) → reranking (Ch.9) →
grounded generation (Ch.10) → orchestration (Ch.11, 13) → evaluation (Ch.16) → governance.

We'll build two practical production tools: a structured configuration object (so architecture
choices are explicit and reviewable, not scattered across code), and a **tiered-routing cost
estimator** that quantifies the Chapter 11/13 idea of "route cheap queries to cheap strategies."


In [61]:
# Chapter 19 -- A production-style configuration object
# Making every architecture choice an explicit, reviewable field (not a scattered set of
# hardcoded constants across a codebase) is itself a production best practice.

class RAGProductionConfig:
    def __init__(
        self,
        chunk_strategy="structure_aware",     # Ch.4
        chunk_size=250,
        embedding_model="self_hosted_bge",    # Ch.5 -- vs. "api_openai"
        retrieval_strategy="hybrid",          # Ch.8 -- "sparse" | "dense" | "hybrid"
        use_reranker=True,                    # Ch.9
        context_token_budget=2000,            # Ch.10
        correction_strategy="crag",           # Ch.11 -- "none" | "self_rag" | "crag" | "adaptive"
        access_control_enabled=True,          # Ch.19 security
        eval_sampling_rate=0.05,              # Ch.16 -- fraction of live queries monitored
    ):
        self.chunk_strategy = chunk_strategy
        self.chunk_size = chunk_size
        self.embedding_model = embedding_model
        self.retrieval_strategy = retrieval_strategy
        self.use_reranker = use_reranker
        self.context_token_budget = context_token_budget
        self.correction_strategy = correction_strategy
        self.access_control_enabled = access_control_enabled
        self.eval_sampling_rate = eval_sampling_rate

    def summary(self):
        lines = [f"  {k} = {v}" for k, v in self.__dict__.items()]
        return "RAGProductionConfig:\n" + "\n".join(lines)


prod_config = RAGProductionConfig(
    embedding_model="api_openai",
    correction_strategy="adaptive",
)
print(prod_config.summary())


RAGProductionConfig:
  chunk_strategy = structure_aware
  chunk_size = 250
  embedding_model = api_openai
  retrieval_strategy = hybrid
  use_reranker = True
  context_token_budget = 2000
  correction_strategy = adaptive
  access_control_enabled = True
  eval_sampling_rate = 0.05


### A tiered-routing cost estimator

Recall Adaptive-RAG (Ch.11) and agentic tiered routing (Ch.13): not every query needs the most
expensive strategy. Let's quantify the actual cost savings of routing queries by complexity instead
of always using the most expensive path.


In [62]:
# Chapter 19 -- Tiered-routing cost estimator (toy, but illustrates real cost-engineering logic)

# Rough relative per-query cost units for each strategy (illustrative, not real pricing)
STRATEGY_COST = {
    "naive_single_pass": 1.0,     # Ch.7 -- one retrieval + one generation call
    "hybrid_reranked": 2.5,       # Ch.8-9 -- two retrieval passes + a reranker pass
    "crag_corrective": 4.0,       # Ch.11 -- adds an evaluator call, sometimes a web search
    "agentic_multistep": 9.0,     # Ch.13 -- multiple tool calls + multiple LLM reasoning steps
}

def estimate_cost(query_distribution, strategy_assignment, n_queries_per_day=10_000):
    """
    query_distribution: dict of {complexity_tier: fraction_of_queries}
    strategy_assignment: dict of {complexity_tier: strategy_name}
    Returns total daily cost in relative units, plus a per-tier breakdown.
    """
    breakdown = {}
    total = 0.0
    for tier, fraction in query_distribution.items():
        strategy = strategy_assignment[tier]
        n_queries = n_queries_per_day * fraction
        tier_cost = n_queries * STRATEGY_COST[strategy]
        breakdown[tier] = {"n_queries": round(n_queries), "strategy": strategy, "cost": round(tier_cost, 1)}
        total += tier_cost
    return {"total_cost": round(total, 1), "breakdown": breakdown}


# A realistic query mix: most queries are simple, few are genuinely complex
query_mix = {"simple": 0.70, "moderate": 0.25, "complex": 0.05}

# Strategy A: always use the most sophisticated strategy, regardless of query complexity
always_agentic = {"simple": "agentic_multistep", "moderate": "agentic_multistep", "complex": "agentic_multistep"}

# Strategy B: tiered routing -- match strategy cost to actual query complexity (Adaptive-RAG style)
tiered_routing = {"simple": "naive_single_pass", "moderate": "hybrid_reranked", "complex": "agentic_multistep"}

cost_a = estimate_cost(query_mix, always_agentic)
cost_b = estimate_cost(query_mix, tiered_routing)

print(f"Always-agentic total daily cost: {cost_a['total_cost']} units")
print(f"Tiered-routing total daily cost: {cost_b['total_cost']} units")
print(f"\nSavings from tiered routing: {round(100 * (1 - cost_b['total_cost']/cost_a['total_cost']), 1)}%")


Always-agentic total daily cost: 90000.0 units
Tiered-routing total daily cost: 17750.0 units

Savings from tiered routing: 80.3%


An 80.3% cost reduction from tiered routing, on a realistic query mix where most questions are
simple. This is a concrete, quantified version of the Adaptive-RAG philosophy from Chapter 11:
routing 70% of queries to a cheap single-pass strategy and reserving the expensive agentic loop
for only the 5% of genuinely complex queries produces massive savings *precisely because* most
real-world query traffic skews toward the simple end — which is also why a system that always
defaults to its most sophisticated architecture is usually solving a problem most of its traffic
doesn't actually have.

> **🏭 Production note:** Real cost engineering also includes **caching** (storing results for
> repeated or near-duplicate queries) and **retry budgets** (Ch.13) as hard backstops against
> runaway cost on pathological queries. The `STRATEGY_COST` values above are illustrative — real
> systems would derive these from actual token counts, API pricing, and infrastructure costs
> specific to their chosen models and vector database.

> **Security and governance, briefly:** `access_control_enabled` in our config isn't just a flag —
> production access control must be enforced **at the retrieval layer itself**, filtering candidate
> chunks by the requesting user's permissions *before* the vector search runs, not just in the
> application layer afterward. And `eval_sampling_rate` reflects a real practice: continuously
> sampling a fraction of live production queries through the Chapter 16 metrics, since a system that
> scored well in development can silently degrade as the corpus, embedding model, or generator
> model changes over time.


### 🧪 Try it yourself

Try a *less* favorable query mix — one where complex queries are much more common (say, 40% complex
instead of 5%). Recompute the tiered-routing savings. Does tiered routing still help as much? What
does this tell you about when the tiered-routing strategy earns its complexity?


In [63]:
# TODO Your turn -- try a much harder query mix (many more complex queries)
harder_query_mix = {"simple": 0.30, "moderate": 0.30, "complex": 0.40}

cost_a_hard = estimate_cost(harder_query_mix, always_agentic)
cost_b_hard = estimate_cost(harder_query_mix, tiered_routing)

print(f"Always-agentic total daily cost: {cost_a_hard['total_cost']} units")
print(f"Tiered-routing total daily cost: {cost_b_hard['total_cost']} units")
savings_hard = round(100 * (1 - cost_b_hard['total_cost']/cost_a_hard['total_cost']), 1)
print(f"\nSavings from tiered routing (harder mix): {savings_hard}%")


Always-agentic total daily cost: 90000.0 units
Tiered-routing total daily cost: 46500.0 units

Savings from tiered routing (harder mix): 48.3%


<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

With 40% of queries now complex (versus the original 5%), tiered-routing savings drop from **80.3%
to 48.3%**. Still a substantial saving, but roughly two-fifths of what it was — a direct, visible
consequence of routing having fewer cheap queries to route cheaply.

This is exactly the point: **tiered routing's payoff scales with how skewed your real query
distribution is toward the simple end.** For a system where most traffic genuinely needs
sophisticated multi-step reasoning, tiered routing still helps (you're never worse off routing
correctly), but the savings are structurally smaller, because there's simply less "cheap" traffic
to capture the discount on. This is a useful sanity check before investing engineering effort in
building a complexity classifier (Chapter 11's Adaptive-RAG) — it's worth first measuring your
*actual* query distribution, since the technique's value is directly proportional to how skewed
that distribution turns out to be, not a fixed percentage you can assume in advance.
</details>


## Chapter 20 — Research Frontiers & Capstone Project

We've built, from first principles, a working (if simplified) version of nearly every major RAG
technique: chunking, embeddings, hybrid search, reranking, CRAG, GraphRAG, agentic loops, RAPTOR,
evaluation metrics, and production cost modeling. Current 2026 research pushes further —
systematizing agentic RAG, hierarchical retrieval interfaces, diversity-aware retrieval,
reasoning-intensive benchmarks (the TREC RAG Track), federated multi-corpus retrieval
(UniversalRAG-style), and knowledge-runtime platforms that treat an index as continuously updating
rather than periodically rebuilt.

This final chapter is a **guided capstone**: a starter scaffold that reuses every component this
notebook built, with TODOs marking exactly where you'd plug in your own corpus and real production
components (a real embedding model, a real vector database, a real LLM) to turn this notebook's
toy implementations into an actual working system.


In [64]:
# Chapter 20 -- Capstone starter scaffold
# This reuses every component built across Chapters 1-19. Replace the TODO-marked lines with
# your own corpus and (optionally) real production components to build an actual working system.

class CapstoneRAGSystem:
    def __init__(self, config: RAGProductionConfig):
        self.config = config
        self.embedder = None
        self.index = None
        self.bm25 = None

    def build_index(self, documents):
        # Step 1: Chunk (Ch.4) -- TODO: swap for your own document loader + chunker if needed
        all_chunks = []
        for doc in documents:
            all_chunks.extend(structure_aware_chunk(doc, chunk_size=self.config.chunk_size))
        texts = [c["text"] for c in all_chunks]
        metas = [{"heading": c["heading"]} for c in all_chunks]

        # Step 2: Embed + Index (Ch.5-6) -- TODO: swap TfidfEmbedder for a real embedding model
        self.embedder = TfidfEmbedder()
        self.embedder.fit(texts)
        self.index = VectorIndex(self.embedder)
        self.index.add(texts, metas)

        # Step 3: Also build a BM25 index if using hybrid retrieval (Ch.8)
        if self.config.retrieval_strategy == "hybrid":
            self.bm25 = BM25()
            self.bm25.fit(texts)

        return len(texts)

    def answer(self, query, k=3, verbose=False):
        # Step 4: Retrieve (Ch.6-8)
        if self.config.retrieval_strategy == "hybrid" and self.bm25:
            fused = hybrid_search(query, self.index.texts, self.bm25, self.embedder, k=k)
            retrieved = [{"text": doc, "score": score} for score, doc in fused]
        else:
            retrieved = self.index.search(query, k=k)

        # Step 5: Rerank (Ch.9) -- optional, per config
        if self.config.use_reranker:
            reranked = rerank(query, [r["text"] for r in retrieved])
            retrieved = [{"text": doc, "score": score} for score, doc in reranked]

        # Step 6: Enforce context budget (Ch.10)
        budgeted, _ = enforce_token_budget(retrieved, max_words=self.config.context_token_budget // 5)

        # Step 7: Build grounded prompt (Ch.10) -- TODO: send this prompt to a real LLM API
        prompt = build_grounded_prompt(query, budgeted, min_relevance_score=None)

        if verbose:
            print("--- PROMPT SENT TO LLM ---")
            print(prompt)
            print("--- END PROMPT ---\n")

        # TODO: replace this stand-in with a real LLM API call, e.g.:
        #   response = anthropic_client.messages.create(model=..., messages=[{"role":"user","content":prompt}])
        if not budgeted:
            return "I don't have enough information to answer this."
        return f"[STAND-IN LLM RESPONSE] Grounded in: \"{budgeted[0]['text'][:100]}...\""


# Wire it all together and test end to end
capstone_config = RAGProductionConfig(retrieval_strategy="hybrid", use_reranker=True)
capstone_system = CapstoneRAGSystem(capstone_config)

n_indexed = capstone_system.build_index([SAMPLE_DOCUMENT])
print(f"Indexed {n_indexed} chunks.\n")

answer = capstone_system.answer("How does structure-aware chunking help technical documents?", verbose=True)
print("ANSWER:", answer)


Indexed 6 chunks.

--- PROMPT SENT TO LLM ---
You are a helpful assistant that answers questions using ONLY the numbered sources below.

RULES:
1. Cite the source number (like [1]) for every claim you make.
2. If the sources do not contain enough information to answer, say "I don't have enough information to answer this" -- do NOT guess or use outside knowledge.
3. Keep the answer concise and directly grounded in the cited sources.

SOURCES:
[1] Structure-aware chunking respects headings and sections, which helps technical documents where
a table or a numbered list should not be split across chunk boundaries.
[2] Chunking determines what unit of text gets retrieved. If chunks are too large, precision drops
because irrelevant text gets pulled in alongside the relevant part. If chunks are too small,
[3] Query-document asymmetry means a question and its answer are rarely phrased the same way, which
is why some embedding models are trained with separate encoders for queries versus document

Notice the top-ranked source in the prompt: `[1] Structure-aware chunking respects headings and
sections, which helps technical documents...` — exactly the passage that answers the question,
retrieved correctly through the full hybrid-search-plus-reranking pipeline, budgeted to fit the
context window, and assembled into a properly grounded, citation-instructed prompt. This is the
complete, real pipeline this notebook has built one piece at a time since Chapter 1 — the only
missing piece is a genuine LLM call at the very end, marked with a `TODO` exactly where it belongs.

### Turning the scaffold into a real system

To move from this notebook to production, the swaps are all localized and don't require touching
the surrounding pipeline logic — this is the modularity payoff from Chapter 3, made concrete:

| Component | Notebook version | Real swap |
|---|---|---|
| Embedder | `TfidfEmbedder` (Ch.5) | `sentence-transformers` or an embedding API |
| Vector index | `VectorIndex` brute-force (Ch.6) | FAISS, Qdrant, Pinecone, or pgvector |
| Reranker | `toy_cross_encoder_score` (Ch.9) | A real cross-encoder (BGE-reranker, Cohere rerank-v3) |
| Generator | `[STAND-IN LLM RESPONSE]` (Ch.20) | An actual LLM API call using `prompt` |
| Evaluation | Word-overlap metrics (Ch.16) | Real `ragas` library with LLM-as-judge |


### Current 2026 research directions

Beyond what we've built, several active research areas push further:

- **Agentic RAG systematization** — moving from ad-hoc agentic loops (Ch.13) toward principled
  frameworks for when and how much agency a retrieval system should have.
- **Hierarchical retrieval interfaces** — extending RAPTOR's multi-granularity idea (Ch.15) into
  standardized interfaces letting an agent (Ch.13) reason explicitly about which level to query.
- **Diversity-aware retrieval** — deliberately returning a diverse set of relevant-but-different
  chunks, rather than several near-duplicate top matches, for broad or ambiguous queries.
- **Reasoning-intensive retrieval benchmarks** — evaluation research (extending Ch.16) shifting
  toward benchmarks testing multi-step reasoning over retrieved evidence, not just single-hop recall.
- **Federated multi-corpus retrieval (UniversalRAG-style)** — routing and retrieving across
  multiple distinct, separately-governed corpora, generalizing Chapter 14's two-way modality router
  to many source types.
- **Knowledge-runtime platforms** — treating a RAG index as a continuously-updating runtime
  integrated with live data sources, rather than requiring periodic re-indexing batches.
- **The TREC RAG Track** — an ongoing community benchmarking effort providing standardized,
  community-vetted test collections specifically for evaluating RAG systems.


### 🧪 Capstone exercise: bring your own corpus

This is the final exercise of the notebook. Build a `CapstoneRAGSystem` over a **new** corpus —
not `SAMPLE_DOCUMENT` — and ask it a question. Suggested scope, echoing the reference document's
own capstone guidance:

1. Write 1-2 short paragraphs (with at least one Markdown heading) about any topic you know well.
2. Index it with `CapstoneRAGSystem`.
3. Ask it a specific question and inspect the full constructed prompt.
4. Ask it a question it *can't* answer from your corpus, and confirm it says so rather than
   fabricating an answer.


In [65]:
# TODO Your turn -- write your own corpus and test the full capstone system end to end

MY_CORPUS_DOCUMENT = """# Coffee Brewing Methods

Pour-over brewing uses a paper or metal filter and gravity to extract coffee slowly, typically
over 2-4 minutes. It produces a clean, bright cup because the filter removes most of the coffee's
natural oils and fine particles.

## French Press

French press brewing steeps coarsely-ground coffee directly in hot water for about 4 minutes,
then separates the grounds using a metal mesh plunger. Because there's no paper filter, more oils
and fine sediment remain in the cup, producing a fuller, heavier body than pour-over.
"""

my_system = CapstoneRAGSystem(RAGProductionConfig(retrieval_strategy="hybrid", use_reranker=True))
my_system.build_index([MY_CORPUS_DOCUMENT])

# A question your corpus CAN answer
answerable = my_system.answer("Why does French press coffee have a heavier body than pour-over?", verbose=True)
print("ANSWER:", answerable)

print("\n" + "="*70 + "\n")

# A question your corpus CANNOT answer
unanswerable = my_system.answer("What temperature should espresso be brewed at?", verbose=False)
print("ANSWER TO UNANSWERABLE QUESTION:", unanswerable)


--- PROMPT SENT TO LLM ---
You are a helpful assistant that answers questions using ONLY the numbered sources below.

RULES:
1. Cite the source number (like [1]) for every claim you make.
2. If the sources do not contain enough information to answer, say "I don't have enough information to answer this" -- do NOT guess or use outside knowledge.
3. Keep the answer concise and directly grounded in the cited sources.

SOURCES:
[1] and fine sediment remain in the cup, producing a fuller, heavier body than pour-over.
[2] French press brewing steeps coarsely-ground coffee directly in hot water for about 4 minutes,
then separates the grounds using a metal mesh plunger. Because there's no paper filter, more oils
[3] Pour-over brewing uses a paper or metal filter and gravity to extract coffee slowly, typically
over 2-4 minutes. It produces a clean, bright cup because the filter removes most of the coffee's
natural oils and fine particles.

QUESTION: Why does French press coffee have a heavier bo

<details>
<summary><b>💡 Solution / discussion (click to expand)</b></summary>

Run this yourself and you'll find something important: the "unanswerable" espresso-temperature
question does **not** correctly say "I don't have enough information" — it confidently returns the
pour-over passage instead, exactly the ungrounded-confidence failure mode this notebook has
surfaced repeatedly (Chapters 2, 7, 11, 13, 16).

Here's the specific, diagnosable reason, and it's a genuinely valuable one to have found in a
capstone: `CapstoneRAGSystem.answer()` calls `build_grounded_prompt(query, budgeted,
min_relevance_score=None)` — the relevance guardrail from Chapter 10 is explicitly disabled. Even
if you enable it, you'd hit a second, subtler problem: `hybrid_search`'s **RRF-fused scores**
(Chapter 8) live on a completely different numeric scale than the **cosine-similarity scores**
(Chapters 5, 7) the `min_relevance_score` threshold was originally tuned against. RRF scores here
top out around 0.033, while our Chapter 7 threshold discussion was calibrated for cosine-similarity
values in the 0.25-0.9 range — so a threshold that made sense for one retrieval strategy is
silently meaningless for another.

This is precisely Chapter 18's "systems layer" failure mode in the wild: **two independently
correct components (RRF fusion, a relevance threshold) become incorrect when composed, because
their assumptions about scale don't match** — and nothing about running either component in
isolation would have revealed the mismatch. A real fix requires either normalizing scores to a
common scale before thresholding, or (more robustly) replacing the raw-score threshold with a
proper reranker-based or CRAG-style relevance judgment (Chapters 9, 11), which don't depend on
any particular upstream scoring scale.

**This is a fitting note to end the notebook on.** Every technique built across these twenty
chapters is a real, working piece — but a production RAG system is only as trustworthy as the
integration between its pieces, and that integration has to be tested explicitly, the same way
this exercise just did, rather than assumed to work because each piece works on its own.
</details>


---

## You've completed RAG Mastery

Across 20 chapters, you built — from first principles, with no external APIs or model downloads —
a working version of nearly every major RAG technique in current use: chunking, TF-IDF embeddings,
brute-force vector search, BM25, hybrid retrieval with RRF, HyDE, cross-encoder-style reranking,
grounded prompting, CRAG, GraphRAG, agentic tool-use loops, modality routing, RAPTOR, MaxSim
scoring, RAGAS-style evaluation, a full taxonomy, a failure-mode checklist, production cost
modeling, and a capstone system tying it all together.

Just as importantly, nearly every chapter's exercise surfaced a **real, observed limitation** —
not a hypothetical one — by actually running the code and checking what it produced, rather than
trusting an assumption about what it should do. That habit is the single most transferable skill
this notebook can offer: production RAG systems fail in exactly these kinds of subtle, specific
ways, and the only reliable way to catch them is to test rigorously, the way every exercise here
just did.

**Where to go next:** swap the notebook's toy components for real ones (the table in this chapter
is your map), read the research papers cited throughout (the reference document's bibliography has
the full list), and build something with a real corpus that matters to you.
